Spotify MPD Fair Benchmark with Sequential, Metadata-Aware, Multi-Interest, and IRRT Experiments

This notebook is the main benchmark for the AML/IRRT project on the Spotify Million Playlist Dataset (MPD).

Its goal is to provide a fair and reproducible comparison between:
- sequential recommendation baselines
- stronger metadata-aware sequential models
- SSL-enhanced sequential models
- efficient long-sequence alternatives
- metadata-aware multi-interest models with label-aware routing
- IRRT-style retrieval-to-reranking pipelines

Included recommendation models
- TopPop
- GRU4Rec
- MetadataSASRec
- MetadataSASRec + CL4SRec-style SSL
- MetadataSASRec + SimDCL-style SSL
- FMLP-Rec-style efficient baseline
- MetadataComiRecSA
- MetadataComiRecSA + label-aware routing + diversity regularization

Included IRRT components
- sparse lexical retrieval from playlist and item metadata
- dense retrieval from learned metadata-aware representations
- hybrid sparse+dense retrieval with reciprocal-rank fusion
- shortlist reranking with trained neural recommenders

Main benchmark properties
- all neural models are evaluated on the same fixed validation and test playlists
- all models use the same filtered item universe
- full-ranking evaluation is implemented in a memory-safe chunked form
- model checkpoints are saved and reused for IRRT evaluation
- results are aggregated across seeds
- Transformer-family models use causal masking and padding masks
- the notebook uses a stratified playlist subsample
- the analysis covers:
  - global ranking quality
  - coverage
  - length-based slices
  - metadata-aware heterogeneity slices
  - target popularity slices
  - unseen-target and hard-continuation slices
  - interest-usage diagnostics
  - candidate recall for retrieval
  - reranked shortlist quality for IRRT

Music-aware metadata used in this notebook
- playlist titles
- track names
- artist names
- album names
- artist and album identifiers
- duration buckets

Scope note

This notebook now contains two connected parts:
1. recommendation benchmarking with full ranking
2. IRRT-style two-stage retrieval and reranking experiments

The main research question is:

> whether metadata-aware sequential and multi-interest models improve both recommendation quality and retrieval-to-reranking quality, especially on heterogeneous and long-tail playlist continuation.

## 1. Imports and experiment configuration

In [1]:
import os
import json
import math
import random
import zipfile
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from time import perf_counter
from tqdm.auto import tqdm
from IPython.display import clear_output, display


In [2]:
PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"
SPOTIFY_DIR = DATA_DIR / "spotify_mpd"
SPOTIFY_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = PROJECT_DIR / "saved_model_checkpoints"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MPD_ZIP_PATH = None

MPD_MAX_SLICES = os.environ.get("MPD_MAX_SLICES", "None")
MPD_MAX_SLICES = None if MPD_MAX_SLICES.lower() == "none" else int(MPD_MAX_SLICES)

MIN_PLAYLIST_LENGTH = int(os.environ.get("MIN_PLAYLIST_LENGTH", "10"))
MAX_PLAYLISTS = os.environ.get("MAX_PLAYLISTS", "None")
MAX_PLAYLISTS = None if MAX_PLAYLISTS.lower() == "none" else int(MAX_PLAYLISTS)

MAX_ITEMS = os.environ.get("MAX_ITEMS", "10000")
MAX_ITEMS = None if MAX_ITEMS.lower() == "none" else int(MAX_ITEMS)
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "100"))

SUBSAMPLE_FRACTION = float(os.environ.get("SUBSAMPLE_FRACTION", "0.60"))
SUBSAMPLE_RANDOM_STATE = int(os.environ.get("SUBSAMPLE_RANDOM_STATE", "2026"))

D_MODEL = int(os.environ.get("D_MODEL", "128"))
N_HEADS = int(os.environ.get("N_HEADS", "4"))
N_LAYERS = int(os.environ.get("N_LAYERS", "2"))
DROPOUT = float(os.environ.get("DROPOUT", "0.1"))
GRU_HIDDEN = int(os.environ.get("GRU_HIDDEN", str(D_MODEL)))
N_INTERESTS = int(os.environ.get("N_INTERESTS", "4"))

FMLP_N_LAYERS = int(os.environ.get("FMLP_N_LAYERS", "2"))
FMLP_DROPOUT = float(os.environ.get("FMLP_DROPOUT", str(DROPOUT)))

BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "128"))
LR = float(os.environ.get("LR", "1e-3"))
WEIGHT_DECAY = float(os.environ.get("WEIGHT_DECAY", "1e-4"))
EPOCHS = int(os.environ.get("EPOCHS", "30"))
PATIENCE = int(os.environ.get("PATIENCE", "3"))

DO_CL4SREC_SSL = os.environ.get("DO_CL4SREC_SSL", "1") == "1"
DO_SIMDCL = os.environ.get("DO_SIMDCL", "1") == "1"
SSL_EPOCHS = int(os.environ.get("SSL_EPOCHS", "10"))
SSL_TAU = float(os.environ.get("SSL_TAU", "0.2"))
SIMDCL_NOISE_STD = float(os.environ.get("SIMDCL_NOISE_STD", "0.1"))

COMIREC_LABEL_AWARE = os.environ.get("COMIREC_LABEL_AWARE", "1") == "1"
COMIREC_DIVERSITY_REG = float(os.environ.get("COMIREC_DIVERSITY_REG", "0.05"))
COMIREC_LABEL_TAU = float(os.environ.get("COMIREC_LABEL_TAU", "1.0"))

TOPK = int(os.environ.get("TOPK", "20"))
EVAL_BATCH_SIZE = int(os.environ.get("EVAL_BATCH_SIZE", "128"))
ITEM_CHUNK_SIZE = int(os.environ.get("ITEM_CHUNK_SIZE", "4096"))

NUM_EVAL_PLAYLISTS = os.environ.get("NUM_EVAL_PLAYLISTS", "None")
NUM_EVAL_PLAYLISTS = None if NUM_EVAL_PLAYLISTS.lower() == "none" else int(NUM_EVAL_PLAYLISTS)

ITEM_TITLE_MAX_TOKENS = int(os.environ.get("ITEM_TITLE_MAX_TOKENS", "8"))
PLAYLIST_TITLE_MAX_TOKENS = int(os.environ.get("PLAYLIST_TITLE_MAX_TOKENS", "8"))
TEXT_VOCAB_MAX = int(os.environ.get("TEXT_VOCAB_MAX", "20000"))

IRRT_RETRIEVAL_CUTOFFS = [50, 100, 200]
IRRT_RERANK_CANDIDATE_SIZE = int(os.environ.get("IRRT_RERANK_CANDIDATE_SIZE", "200"))
IRRT_RERANK_TOPK = int(os.environ.get("IRRT_RERANK_TOPK", str(TOPK)))
IRRT_HISTORY_TEXT_LAST_N = int(os.environ.get("IRRT_HISTORY_TEXT_LAST_N", "20"))
IRRT_TITLE_QUERY_WEIGHT = float(os.environ.get("IRRT_TITLE_QUERY_WEIGHT", "2.0"))
IRRT_HISTORY_QUERY_WEIGHT = float(os.environ.get("IRRT_HISTORY_QUERY_WEIGHT", "1.0"))
IRRT_HYBRID_DENSE_WEIGHT = float(os.environ.get("IRRT_HYBRID_DENSE_WEIGHT", "1.0"))
IRRT_HYBRID_SPARSE_WEIGHT = float(os.environ.get("IRRT_HYBRID_SPARSE_WEIGHT", "1.0"))

EXACT_DENSE_PIPELINE_NAME = "ExactDenseTopM"
HYBRID_PIPELINE_NAME = "HybridRRF"

SEEDS = [42, 43, 44]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert MIN_PLAYLIST_LENGTH >= 3, "Need at least 3 items per playlist for train/valid/test split."
assert 0.0 < SUBSAMPLE_FRACTION <= 1.0
assert IRRT_RERANK_CANDIDATE_SIZE >= IRRT_RERANK_TOPK, "Candidate size must be >= rerank top-k."

EXPERIMENT_CONFIG = {
    "MAX_SEQ_LEN": MAX_SEQ_LEN,
    "D_MODEL": D_MODEL,
    "N_HEADS": N_HEADS,
    "N_LAYERS": N_LAYERS,
    "DROPOUT": DROPOUT,
    "GRU_HIDDEN": GRU_HIDDEN,
    "N_INTERESTS": N_INTERESTS,
    "FMLP_N_LAYERS": FMLP_N_LAYERS,
    "FMLP_DROPOUT": FMLP_DROPOUT,
    "ITEM_TITLE_MAX_TOKENS": ITEM_TITLE_MAX_TOKENS,
    "PLAYLIST_TITLE_MAX_TOKENS": PLAYLIST_TITLE_MAX_TOKENS,
}

print("SPOTIFY_DIR:", SPOTIFY_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("MPD_MAX_SLICES:", MPD_MAX_SLICES)
print("Filtering:", dict(
    MIN_PLAYLIST_LENGTH=MIN_PLAYLIST_LENGTH,
    MAX_PLAYLISTS=MAX_PLAYLISTS,
    MAX_ITEMS=MAX_ITEMS,
    MAX_SEQ_LEN=MAX_SEQ_LEN,
))
print("Subsample:", dict(
    SUBSAMPLE_FRACTION=SUBSAMPLE_FRACTION,
    SUBSAMPLE_RANDOM_STATE=SUBSAMPLE_RANDOM_STATE,
))
print("Model:", dict(
    D_MODEL=D_MODEL,
    N_HEADS=N_HEADS,
    N_LAYERS=N_LAYERS,
    DROPOUT=DROPOUT,
    GRU_HIDDEN=GRU_HIDDEN,
    N_INTERESTS=N_INTERESTS,
    FMLP_N_LAYERS=FMLP_N_LAYERS,
))
print("Train:", dict(
    BATCH_SIZE=BATCH_SIZE,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    EPOCHS=EPOCHS,
    PATIENCE=PATIENCE,
))
print("SSL:", dict(
    DO_CL4SREC_SSL=DO_CL4SREC_SSL,
    DO_SIMDCL=DO_SIMDCL,
    SSL_EPOCHS=SSL_EPOCHS,
    SSL_TAU=SSL_TAU,
    SIMDCL_NOISE_STD=SIMDCL_NOISE_STD,
))
print("ComiRec:", dict(
    COMIREC_LABEL_AWARE=COMIREC_LABEL_AWARE,
    COMIREC_DIVERSITY_REG=COMIREC_DIVERSITY_REG,
    COMIREC_LABEL_TAU=COMIREC_LABEL_TAU,
))
print("Text:", dict(
    ITEM_TITLE_MAX_TOKENS=ITEM_TITLE_MAX_TOKENS,
    PLAYLIST_TITLE_MAX_TOKENS=PLAYLIST_TITLE_MAX_TOKENS,
    TEXT_VOCAB_MAX=TEXT_VOCAB_MAX,
))
print("Eval:", dict(
    TOPK=TOPK,
    EVAL_BATCH_SIZE=EVAL_BATCH_SIZE,
    ITEM_CHUNK_SIZE=ITEM_CHUNK_SIZE,
    NUM_EVAL_PLAYLISTS=NUM_EVAL_PLAYLISTS,
    SEEDS=SEEDS,
))
print("IRRT:", dict(
    IRRT_RETRIEVAL_CUTOFFS=IRRT_RETRIEVAL_CUTOFFS,
    IRRT_RERANK_CANDIDATE_SIZE=IRRT_RERANK_CANDIDATE_SIZE,
    IRRT_RERANK_TOPK=IRRT_RERANK_TOPK,
    IRRT_HISTORY_TEXT_LAST_N=IRRT_HISTORY_TEXT_LAST_N,
    IRRT_TITLE_QUERY_WEIGHT=IRRT_TITLE_QUERY_WEIGHT,
    IRRT_HISTORY_QUERY_WEIGHT=IRRT_HISTORY_QUERY_WEIGHT,
    IRRT_HYBRID_DENSE_WEIGHT=IRRT_HYBRID_DENSE_WEIGHT,
    IRRT_HYBRID_SPARSE_WEIGHT=IRRT_HYBRID_SPARSE_WEIGHT,
    EXACT_DENSE_PIPELINE_NAME=EXACT_DENSE_PIPELINE_NAME,
    HYBRID_PIPELINE_NAME=HYBRID_PIPELINE_NAME,
))
print("DEVICE:", DEVICE)

SPOTIFY_DIR: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd
MODEL_DIR: C:\Users\sokos\DataspellProjects\MML_Project\saved_model_checkpoints
MPD_MAX_SLICES: None
Filtering: {'MIN_PLAYLIST_LENGTH': 10, 'MAX_PLAYLISTS': None, 'MAX_ITEMS': 10000, 'MAX_SEQ_LEN': 100}
Subsample: {'SUBSAMPLE_FRACTION': 0.6, 'SUBSAMPLE_RANDOM_STATE': 2026}
Model: {'D_MODEL': 128, 'N_HEADS': 4, 'N_LAYERS': 2, 'DROPOUT': 0.1, 'GRU_HIDDEN': 128, 'N_INTERESTS': 4, 'FMLP_N_LAYERS': 2}
Train: {'BATCH_SIZE': 128, 'LR': 0.001, 'WEIGHT_DECAY': 0.0001, 'EPOCHS': 30, 'PATIENCE': 3}
SSL: {'DO_CL4SREC_SSL': True, 'DO_SIMDCL': True, 'SSL_EPOCHS': 10, 'SSL_TAU': 0.2, 'SIMDCL_NOISE_STD': 0.1}
ComiRec: {'COMIREC_LABEL_AWARE': True, 'COMIREC_DIVERSITY_REG': 0.05, 'COMIREC_LABEL_TAU': 1.0}
Text: {'ITEM_TITLE_MAX_TOKENS': 8, 'PLAYLIST_TITLE_MAX_TOKENS': 8, 'TEXT_VOCAB_MAX': 20000}
Eval: {'TOPK': 20, 'EVAL_BATCH_SIZE': 128, 'ITEM_CHUNK_SIZE': 4096, 'NUM_EVAL_PLAYLISTS': None, 'SEEDS': [42, 43, 44]}
IRRT: {'IRRT_RETR

## 2. Dataset discovery and raw MPD loading

In this section, the notebook:
- extracts the local MPD archive,
- locates MPD slice files,
- checks that the dataset is available,
- and prepares the raw slice list for parsing.

In [3]:
def extract_local_zip(zip_path: Optional[Path], out_dir: Path) -> None:
    if zip_path is None:
        print("MPD_ZIP_PATH is None (assuming the dataset is already extracted).")
        return
    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip archive not found: {zip_path}")
    print(f"Extracting {zip_path} -> {out_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
    print("Extraction complete.")

extract_local_zip(MPD_ZIP_PATH, SPOTIFY_DIR)

slice_files = sorted(SPOTIFY_DIR.rglob("mpd.slice.*.json"))
if MPD_MAX_SLICES is not None:
    slice_files = slice_files[:MPD_MAX_SLICES]

print("Found slice files:", len(slice_files))
if len(slice_files) == 0:
    raise FileNotFoundError(
        "No MPD slice files were found. Put extracted files into data/spotify_mpd/ "
        "so that files like 'mpd.slice.0-999.json' are visible."
    )
print("First slice:", slice_files[0])
if len(slice_files) > 1:
    print("Last slice:", slice_files[-1])

MPD_ZIP_PATH is None (assuming the dataset is already extracted).
Found slice files: 1000
First slice: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd\mpd.slice.0-999.json
Last slice: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd\mpd.slice.999000-999999.json


## 3. Parse playlist-track interactions

Here I convert MPD slices into a simple interaction table with:
- playlist ID,
- track ID,
- and track position inside the playlist.

This is the raw sequential interaction format used by the rest of the pipeline.

In [4]:
def safe_str(x) -> str:
    return "" if x is None else str(x)


def duration_bucket(ms: int) -> str:
    if ms <= 0:
        return "unk"
    sec = ms / 1000.0
    if sec < 60:
        return "<1m"
    if sec < 120:
        return "1-2m"
    if sec < 180:
        return "2-3m"
    if sec < 240:
        return "3-4m"
    if sec < 300:
        return "4-5m"
    return "5m+"


def parse_mpd_slices_to_sequences(files: List[Path]) -> pd.DataFrame:
    rows = []
    for path in files:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for pl in data.get("playlists", []):
            pid = int(pl["pid"])
            playlist_name = safe_str(pl.get("name")).strip().lower()

            for pos, tr in enumerate(pl.get("tracks", [])):
                track_uri = tr.get("track_uri")
                if track_uri is None:
                    continue

                duration_ms = int(tr.get("duration_ms", 0) or 0)

                rows.append({
                    "pid": pid,
                    "playlist_name": playlist_name,
                    "track_uri": safe_str(track_uri),
                    "track_name": safe_str(tr.get("track_name")).strip().lower(),
                    "artist_name": safe_str(tr.get("artist_name")).strip().lower(),
                    "album_name": safe_str(tr.get("album_name")).strip().lower(),
                    "artist_uri": safe_str(tr.get("artist_uri")),
                    "album_uri": safe_str(tr.get("album_uri")),
                    "duration_ms": duration_ms,
                    "duration_bucket": duration_bucket(duration_ms),
                    "pos": int(pos),
                })
    return pd.DataFrame(rows)

interactions = parse_mpd_slices_to_sequences(slice_files)
print("Parsed interactions:", interactions.shape)
display(interactions.head())

Parsed interactions: (66346428, 11)


,pid,playlist_name,track_uri,track_name,artist_name,album_name,artist_uri,album_uri,duration_ms,duration_bucket,pos
0,0,throwbacks,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,lose control (feat. ciara & fat man scoop),missy elliott,the cookbook,spotify:artist:2wIVse2owClT7go1WT98tk,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,226863,3-4m,0
1,0,throwbacks,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,toxic,britney spears,in the zone,spotify:artist:26dSoYclwsYLMAKD3tpOr4,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,198800,3-4m,1
2,0,throwbacks,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,crazy in love,beyoncé,dangerously in love (alben für die ewigkeit),spotify:artist:6vWDO969PvNqNYHIOW5v0m,spotify:album:25hVFAxTlDvXbx2X2QkUkE,235933,3-4m,2
3,0,throwbacks,spotify:track:1AWQoqb9bSvzTjaLralEkT,rock your body,justin timberlake,justified,spotify:artist:31TPClRtHm23RisEBtV3X7,spotify:album:6QPkyl04rXwTGlGlcYaRoW,267266,4-5m,3
4,0,throwbacks,spotify:track:1lzr43nnXAijIGYnCT8M8H,it wasn't me,shaggy,hot shot,spotify:artist:5EvFsr3kj42KNv97ZEnqij,spotify:album:6NmFmPX56pcLBOFMhIiKvF,227600,3-4m,4


## 4. Interaction cleaning and dataset filtering

This stage:
- sorts interactions by playlist and position,
- removes exact duplicates,
- filters out very short playlists,
- optionally caps the number of playlists and items,
- and applies a stratified subsample by playlist length.

In [5]:
interactions = interactions.copy()
interactions["pid"] = interactions["pid"].astype(int)
interactions["track_uri"] = interactions["track_uri"].astype(str)
interactions["playlist_name"] = interactions["playlist_name"].fillna("").astype(str)
interactions["track_name"] = interactions["track_name"].fillna("").astype(str)
interactions["artist_name"] = interactions["artist_name"].fillna("").astype(str)
interactions["album_name"] = interactions["album_name"].fillna("").astype(str)
interactions["artist_uri"] = interactions["artist_uri"].fillna("").astype(str)
interactions["album_uri"] = interactions["album_uri"].fillna("").astype(str)
interactions["duration_ms"] = pd.to_numeric(interactions["duration_ms"], errors="coerce").fillna(0).astype(int)
interactions["duration_bucket"] = interactions["duration_bucket"].fillna("unk").astype(str)
interactions["pos"] = interactions["pos"].astype(int)

interactions = (
    interactions
    .sort_values(["pid", "pos"])
    .drop_duplicates(subset=["pid", "track_uri", "pos"])
    .reset_index(drop=True)
)
print("Cleaned interactions:", interactions.shape)
display(interactions.head())

Cleaned interactions: (66346428, 11)


,pid,playlist_name,track_uri,track_name,artist_name,album_name,artist_uri,album_uri,duration_ms,duration_bucket,pos
0,0,throwbacks,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,lose control (feat. ciara & fat man scoop),missy elliott,the cookbook,spotify:artist:2wIVse2owClT7go1WT98tk,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,226863,3-4m,0
1,0,throwbacks,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,toxic,britney spears,in the zone,spotify:artist:26dSoYclwsYLMAKD3tpOr4,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,198800,3-4m,1
2,0,throwbacks,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,crazy in love,beyoncé,dangerously in love (alben für die ewigkeit),spotify:artist:6vWDO969PvNqNYHIOW5v0m,spotify:album:25hVFAxTlDvXbx2X2QkUkE,235933,3-4m,2
3,0,throwbacks,spotify:track:1AWQoqb9bSvzTjaLralEkT,rock your body,justin timberlake,justified,spotify:artist:31TPClRtHm23RisEBtV3X7,spotify:album:6QPkyl04rXwTGlGlcYaRoW,267266,4-5m,3
4,0,throwbacks,spotify:track:1lzr43nnXAijIGYnCT8M8H,it wasn't me,shaggy,hot shot,spotify:artist:5EvFsr3kj42KNv97ZEnqij,spotify:album:6NmFmPX56pcLBOFMhIiKvF,227600,3-4m,4


In [6]:
def filter_playlists_by_min_length(df: pd.DataFrame, min_len: int) -> pd.DataFrame:
    cnt = df.groupby("pid").size()
    keep = cnt[cnt >= min_len].index
    return df[df["pid"].isin(keep)].copy()


def cap_top_playlists(df: pd.DataFrame, max_playlists: Optional[int]) -> pd.DataFrame:
    if max_playlists is None or df["pid"].nunique() <= max_playlists:
        return df
    top_playlists = (
        df.groupby("pid")
        .size()
        .sort_values(ascending=False)
        .head(max_playlists)
        .index
    )
    return df[df["pid"].isin(top_playlists)].copy()


def cap_top_items(df: pd.DataFrame, max_items: Optional[int]) -> pd.DataFrame:
    if max_items is None or df["track_uri"].nunique() <= max_items:
        return df
    top_items = (
        df.groupby("track_uri")
        .size()
        .sort_values(ascending=False)
        .head(max_items)
        .index
    )
    return df[df["track_uri"].isin(top_items)].copy()


def build_playlist_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = (
        df.groupby("pid")
        .size()
        .rename("playlist_len")
        .reset_index()
    )

    bins = [9, 19, 39, 79, 199, np.inf]
    labels = ["10-19", "20-39", "40-79", "80-199", "200+"]

    meta["len_bin"] = pd.cut(
        meta["playlist_len"],
        bins=bins,
        labels=labels,
        right=True,
    )
    return meta


def add_length_bin(df: pd.DataFrame) -> pd.DataFrame:
    meta = build_playlist_meta(df)
    out = df.merge(meta, on="pid", how="left")
    return out


def stratified_playlist_subsample(
    df: pd.DataFrame,
    fraction: float,
    random_state: int,
) -> pd.DataFrame:
    meta = build_playlist_meta(df)

    sampled_pids: List[int] = []
    rng = np.random.RandomState(random_state)

    for _, g in meta.groupby("len_bin", observed=False):
        if len(g) == 0:
            continue

        n_take = max(1, int(round(len(g) * fraction)))
        n_take = min(n_take, len(g))

        sampled = g.sample(
            n=n_take,
            random_state=int(rng.randint(0, 10**9)),
        )["pid"].tolist()

        sampled_pids.extend(sampled)

    sampled_pids = sorted(set(sampled_pids))
    return df[df["pid"].isin(sampled_pids)].copy()

In [7]:
df = interactions.copy()
df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)
df = cap_top_playlists(df, MAX_PLAYLISTS)
df = cap_top_items(df, MAX_ITEMS)
df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)

full_meta = build_playlist_meta(df)
full_playlist_meta = (
    full_meta.groupby("len_bin", observed=False)
    .size()
    .rename("n_full")
    .reset_index()
)

df = stratified_playlist_subsample(df, SUBSAMPLE_FRACTION, SUBSAMPLE_RANDOM_STATE)
df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)
df = add_length_bin(df)
df = df.sort_values(["pid", "pos"]).reset_index(drop=True)

sub_meta = build_playlist_meta(df)
sub_playlist_meta = (
    sub_meta.groupby("len_bin", observed=False)
    .size()
    .rename("n_subsample")
    .reset_index()
)

display(
    full_playlist_meta
    .merge(sub_playlist_meta, on="len_bin", how="outer")
    .fillna(0)
)

print("Interactions after stratified subsample:", df.shape)
print("Unique playlists after subsample:", df["pid"].nunique())
print("Unique tracks after subsample:", df["track_uri"].nunique())

,len_bin,n_full,n_subsample
0,10-19,184983,110990
1,20-39,237617,142570
2,40-79,207874,124724
3,80-199,120973,72584
4,200+,1999,1199


Interactions after stratified subsample: (21183227, 13)
Unique playlists after subsample: 452067
Unique tracks after subsample: 10000


## 5. Remap playlists and tracks to integer IDs

The models operate on compact integer IDs rather than raw playlist or track identifiers.

This section:
- maps playlists to user IDs,
- maps tracks to item IDs,
- and prepares the final interaction table used for sequence modeling.

In [8]:
def remap_series_to_int(s: pd.Series) -> Tuple[pd.Series, Dict[str, int]]:
    uniq = s.unique()
    mapping = {v: i + 1 for i, v in enumerate(uniq)}
    return s.map(mapping).astype(np.int32), mapping


df["user_id"], playlist_map = remap_series_to_int(df["pid"].astype(str))
df["item_id"], item_map = remap_series_to_int(df["track_uri"])
df = df.drop(columns=["track_uri"])

n_users = int(df["user_id"].max())
n_items = int(df["item_id"].max())

print("Filtered interactions:", df.shape)
print("Playlists:", n_users, "| Tracks:", n_items)
display(df.head())

Filtered interactions: (21183227, 14)
Playlists: 452067 | Tracks: 10000


,pid,playlist_name,track_name,artist_name,album_name,artist_uri,album_uri,duration_ms,duration_bucket,pos,playlist_len,len_bin,user_id,item_id
0,3,mat,when did your heart go missing?,rooney,calling the world,spotify:artist:78inRkIIF0n9R9I2HPoWP7,spotify:album:7kHRGtUgh8UTZhN3A8eJF1,212926,3-4m,23,13,10-19,1,1
1,3,mat,two weeks,grizzly bear,veckatimest,spotify:artist:2Jv5eshHtLycR6R8KQCdc4,spotify:album:6FIFqclBriPCb0SjWDaHIk,243160,4-5m,26,13,10-19,1,2
2,3,mat,campus,vampire weekend,vampire weekend,spotify:artist:5BvJzeQpmsdsFp4HGUYUEx,spotify:album:5oXBmKbyJeQftWMo87cQ9F,176466,2-3m,28,13,10-19,1,3
3,3,mat,trip switch,nothing but thieves,nothing but thieves,spotify:artist:1kDGbuxWknIKx4FlgWxiSp,spotify:album:0vBxqUHwetFn5jCwAKX7Uk,181146,3-4m,31,13,10-19,1,4
4,3,mat,heart it races - dr dog version,dr. dog,heart it races,spotify:artist:4mLJ3XfOM5FPjSAWdQ2Jk7,spotify:album:6wY9YCNf0EpuogJgcPLY0W,233128,3-4m,32,13,10-19,1,5


In [9]:
item_freq = df["item_id"].value_counts()
print("Most frequent item_id:", int(item_freq.index[0]), "count:", int(item_freq.iloc[0]))
print("Unique items:", df["item_id"].nunique())

Most frequent item_id: 275 count: 27556
Unique items: 10000


## 5.1 Build item and playlist metadata tables

In this section, I derive compact metadata tables that can be used for:
- metadata-aware slicing,
- music-aware diversity analysis,
- artist / album based sequence features,
- and later metadata-aware model extensions.

In [10]:
def tokenize_text_simple(text: str) -> List[str]:
    text = (text or "").strip().lower()
    if not text:
        return []
    return [tok for tok in text.replace("/", " ").replace("-", " ").split() if tok]


def build_token_vocab(texts: List[str], min_freq: int = 1, max_vocab: Optional[int] = 20000) -> Dict[str, int]:
    cnt = Counter()
    for text in texts:
        cnt.update(tokenize_text_simple(text))

    items = [(tok, freq) for tok, freq in cnt.items() if freq >= min_freq]
    items.sort(key=lambda x: (-x[1], x[0]))

    if max_vocab is not None:
        items = items[:max_vocab]

    vocab = {"<pad>": 0, "<unk>": 1}
    for tok, _ in items:
        vocab[tok] = len(vocab)
    return vocab


def encode_text_tokens(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    toks = tokenize_text_simple(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in toks[:max_len]]
    if len(ids) < max_len:
        ids = ids + [vocab["<pad>"]] * (max_len - len(ids))
    return ids


def remap_with_pad(values: pd.Series) -> Tuple[pd.Series, Dict[str, int]]:
    uniq = sorted(values.fillna("").astype(str).unique().tolist())
    mapping = {"": 0}
    for v in uniq:
        if v == "":
            continue
        mapping[v] = len(mapping)
    return values.fillna("").astype(str).map(mapping).astype(np.int32), mapping

item_meta = (
    df.groupby("item_id", as_index=False)
    .agg(
        track_name=("track_name", "first"),
        artist_name=("artist_name", "first"),
        album_name=("album_name", "first"),
        artist_uri=("artist_uri", "first"),
        album_uri=("album_uri", "first"),
        duration_bucket=("duration_bucket", "first"),
    )
)

playlist_meta = (
    df.groupby("user_id", as_index=False)
    .agg(
        playlist_name=("playlist_name", "first"),
    )
)

item_meta["artist_id"], artist_map = remap_with_pad(item_meta["artist_uri"])
item_meta["album_id"], album_map = remap_with_pad(item_meta["album_uri"])
item_meta["duration_id"], duration_map = remap_with_pad(item_meta["duration_bucket"])

track_title_vocab = build_token_vocab(item_meta["track_name"].fillna("").astype(str).tolist(), max_vocab=TEXT_VOCAB_MAX)
playlist_title_vocab = build_token_vocab(playlist_meta["playlist_name"].fillna("").astype(str).tolist(), max_vocab=TEXT_VOCAB_MAX)

item_meta["track_title_token_ids"] = item_meta["track_name"].map(
    lambda x: encode_text_tokens(x, track_title_vocab, ITEM_TITLE_MAX_TOKENS)
)

playlist_meta["playlist_title_token_ids"] = playlist_meta["playlist_name"].map(
    lambda x: encode_text_tokens(x, playlist_title_vocab, PLAYLIST_TITLE_MAX_TOKENS)
)

item_feature_table = item_meta[[
    "item_id",
    "artist_id",
    "album_id",
    "duration_id",
    "track_title_token_ids",
    "track_name",
    "artist_name",
    "album_name",
]].copy()

playlist_feature_table = playlist_meta[[
    "user_id",
    "playlist_title_token_ids",
    "playlist_name",
]].copy()

itemid_to_artist_id = dict(zip(item_feature_table["item_id"], item_feature_table["artist_id"]))
itemid_to_album_id = dict(zip(item_feature_table["item_id"], item_feature_table["album_id"]))
itemid_to_duration_id = dict(zip(item_feature_table["item_id"], item_feature_table["duration_id"]))
itemid_to_track_title_tokens = dict(zip(item_feature_table["item_id"], item_feature_table["track_title_token_ids"]))

userid_to_playlist_title_tokens = dict(zip(playlist_feature_table["user_id"], playlist_feature_table["playlist_title_token_ids"]))

itemid_to_artist = dict(zip(item_meta["item_id"], item_meta["artist_uri"]))
itemid_to_album = dict(zip(item_meta["item_id"], item_meta["album_uri"]))
itemid_to_duration_bucket = dict(zip(item_meta["item_id"], item_meta["duration_bucket"]))
itemid_to_track_name = dict(zip(item_meta["item_id"], item_meta["track_name"]))
itemid_to_artist_name = dict(zip(item_meta["item_id"], item_meta["artist_name"]))
itemid_to_album_name = dict(zip(item_meta["item_id"], item_meta["album_name"]))
userid_to_playlist_title = dict(zip(playlist_meta["user_id"], playlist_meta["playlist_name"]))

N_ARTISTS = int(item_meta["artist_id"].max()) + 1
N_ALBUMS = int(item_meta["album_id"].max()) + 1
N_DURATIONS = int(item_meta["duration_id"].max()) + 1
N_TRACK_TITLE_TOKENS = len(track_title_vocab)
N_PLAYLIST_TITLE_TOKENS = len(playlist_title_vocab)

display(item_feature_table.head())
display(playlist_feature_table.head())

print("Artists vocab:", N_ARTISTS)
print("Albums vocab:", N_ALBUMS)
print("Duration vocab:", N_DURATIONS)
print("Track title vocab:", N_TRACK_TITLE_TOKENS)
print("Playlist title vocab:", N_PLAYLIST_TITLE_TOKENS)

,item_id,artist_id,album_id,duration_id,track_title_token_ids,track_name,artist_name,album_name
0,1,2542,5793,3,"[68, 374, 24, 52, 37, 5462, 0, 0]",when did your heart go missing?,rooney,calling the world
1,2,843,4753,4,"[198, 1652, 0, 0, 0, 0, 0, 0]",two weeks,grizzly bear,veckatimest
2,3,1859,4409,2,"[3637, 0, 0, 0, 0, 0, 0, 0]",campus,vampire weekend,vampire weekend
3,4,630,716,3,"[2664, 2602, 0, 0, 0, 0, 0, 0]",trip switch,nothing but thieves,nothing but thieves
4,5,1716,5339,3,"[52, 11, 1134, 4144, 601, 14, 0, 0]",heart it races - dr dog version,dr. dog,heart it races


,user_id,playlist_title_token_ids,playlist_name
0,1,"[6908, 0, 0, 0, 0, 0, 0, 0]",mat
1,2,"[34, 0, 0, 0, 0, 0, 0, 0]",wedding
2,3,"[20, 0, 0, 0, 0, 0, 0, 0]",2017
3,4,"[702, 0, 0, 0, 0, 0, 0, 0]",bop
4,5,"[962, 0, 0, 0, 0, 0, 0, 0]",abby


Artists vocab: 2789
Albums vocab: 5987
Duration vocab: 7
Track title vocab: 7215
Playlist title vocab: 16087


## 5.2 Build lexical retrieval features for IRRT

This section builds sparse lexical representations for:
- playlist retrieval queries
- item retrieval documents

These features are used in the IRRT experiments for:
- sparse lexical retrieval
- hybrid sparse+dense retrieval
- retrieval candidate recall evaluation
- shortlist reranking evaluation

In [11]:
def build_item_lexical_text(row: pd.Series) -> str:
    parts = [
        str(row.get("track_name", "")),
        str(row.get("artist_name", "")),
        str(row.get("album_name", "")),
    ]
    return " ".join([p.strip().lower() for p in parts if str(p).strip()])


item_meta["lexical_text"] = item_meta.apply(build_item_lexical_text, axis=1)
playlist_meta["playlist_title_text"] = playlist_meta["playlist_name"].fillna("").astype(str).str.lower()

itemid_to_lexical_text = dict(zip(item_meta["item_id"], item_meta["lexical_text"]))
userid_to_playlist_title_text = dict(zip(playlist_meta["user_id"], playlist_meta["playlist_title_text"]))

def build_doc_freq(texts: List[str]) -> Counter:
    df_counter = Counter()
    for text in texts:
        toks = set(tokenize_text_simple(text))
        df_counter.update(toks)
    return df_counter


LEXICAL_DF = build_doc_freq(item_meta["lexical_text"].tolist())
LEXICAL_N_DOCS = max(1, len(item_meta))


def text_to_tfidf_counter(
    text: str,
    df_counter: Counter,
    n_docs: int,
) -> Dict[str, float]:
    toks = tokenize_text_simple(text)
    tf = Counter(toks)
    out: Dict[str, float] = {}

    for tok, freq in tf.items():
        df_val = df_counter.get(tok, 0)
        idf = math.log((1.0 + n_docs) / (1.0 + df_val)) + 1.0
        out[tok] = float(freq) * idf

    return out


itemid_to_sparse_vec = {
    item_id: text_to_tfidf_counter(itemid_to_lexical_text.get(item_id, ""), LEXICAL_DF, LEXICAL_N_DOCS)
    for item_id in range(1, n_items + 1)
}

print("Lexical sparse representations prepared.")
print("Example item lexical text:", item_meta.loc[0, "lexical_text"] if len(item_meta) > 0 else "")

Lexical sparse representations prepared.
Example item lexical text: when did your heart go missing? rooney calling the world


## 6. Build sequential train/validation/test splits

For each playlist, I build an ordered item sequence and use a simple last-item protocol:
- train sequence: all items except the last two,
- validation sequence: all items except the last one,
- test sequence: full sequence with the last item as the target.

This keeps the setup consistent with next-item prediction.

In [12]:
@dataclass
class SeqData:
    user_seqs: Dict[int, List[int]]
    n_users: int
    n_items: int


def right_pad(seq, max_len: int, pad=0):
    if len(seq) >= max_len:
        return list(seq[-max_len:])
    return list(seq) + [pad for _ in range(max_len - len(seq))]


user_seqs = df.groupby("user_id")["item_id"].apply(list).to_dict()
seqdata = SeqData(user_seqs=user_seqs, n_users=n_users, n_items=n_items)


def split_train_valid_test(seqdata: SeqData):
    train_seqs, valid_seqs, test_seqs = {}, {}, {}
    for u, seq in seqdata.user_seqs.items():
        if len(seq) < 3:
            continue
        train_seqs[u] = seq[:-2]
        valid_seqs[u] = seq[:-1]
        test_seqs[u] = seq[:]
    return train_seqs, valid_seqs, test_seqs


train_seqs, valid_seqs, test_seqs = split_train_valid_test(seqdata)
print("Playlists with train/valid/test sequences:", len(train_seqs))

Playlists with train/valid/test sequences: 452067


## 7. Freeze evaluation users and evaluation cases

To make model comparison fair, I freeze:
- validation playlists,
- test playlists,
- and the exact sequence histories used during evaluation.

This ensures that all models are scored on the same cases.

In [13]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_fixed_eval_users(eval_seqs: Dict[int, List[int]], num_eval_users: Optional[int], seed: int) -> List[int]:
    users = list(eval_seqs.keys())
    rng = random.Random(seed)
    rng.shuffle(users)
    if num_eval_users is None:
        return users
    return users[:min(num_eval_users, len(users))]


@dataclass
class EvalCases:
    users: List[int]
    histories: np.ndarray
    targets: np.ndarray


def build_eval_cases(
    eval_seqs: Dict[int, List[int]],
    users: List[int],
    max_len: int,
) -> EvalCases:
    histories, targets = [], []

    for u in users:
        seq = eval_seqs[u]
        hist = right_pad(seq[:-1], max_len)
        target = seq[-1]

        histories.append(hist)
        targets.append(target)

    return EvalCases(
        users=users,
        histories=np.asarray(histories, dtype=np.int64),
        targets=np.asarray(targets, dtype=np.int64),
    )


EVAL_USER_SEED = 2026

valid_users_fixed = build_fixed_eval_users(valid_seqs, NUM_EVAL_PLAYLISTS, EVAL_USER_SEED)
test_users_fixed = build_fixed_eval_users(test_seqs, NUM_EVAL_PLAYLISTS, EVAL_USER_SEED)

valid_cases = build_eval_cases(valid_seqs, valid_users_fixed, MAX_SEQ_LEN)
test_cases = build_eval_cases(test_seqs, test_users_fixed, MAX_SEQ_LEN)

print("Validation cases:", len(valid_cases.users))
print("Test cases:", len(test_cases.users))

Validation cases: 452067
Test cases: 452067


## 8. Build analysis metadata and subset definitions

Besides global test metrics, the notebook evaluates targeted subsets that are consistent with the no-repeat playlist continuation setting.

This section creates:
- length-based slices,
- metadata-aware heterogeneity slices,
- target popularity slices,
- unseen-target slices,
- and hard-continuation slices.

In [14]:
def safe_norm_entropy_from_counts(counts: np.ndarray) -> float:
    counts = counts[counts > 0].astype(np.float64)
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    ent = -(p * np.log(p + 1e-12)).sum()
    max_ent = math.log(len(p))
    return 0.0 if max_ent <= 0 else float(ent / max_ent)


def compute_history_diversity_features(
    seq: List[int],
    artist_map: Dict[int, str],
    album_map: Dict[int, str],
    duration_bucket_map: Dict[int, str],
) -> Dict[str, float]:
    hist = list(seq)
    L = len(hist)

    if L == 0:
        return {
            "hist_len": 0.0,
            "item_unique_ratio": 0.0,
            "artist_unique_ratio": 0.0,
            "album_unique_ratio": 0.0,
            "item_entropy": 0.0,
            "artist_entropy": 0.0,
            "duration_entropy": 0.0,
            "diversity_score": 0.0,
        }

    item_counts = np.array(list(Counter(hist).values()), dtype=np.int64)

    artists = [artist_map.get(x, "") for x in hist]
    albums = [album_map.get(x, "") for x in hist]
    durations = [duration_bucket_map.get(x, "unk") for x in hist]

    artist_counts = np.array(list(Counter(artists).values()), dtype=np.int64)
    duration_counts = np.array(list(Counter(durations).values()), dtype=np.int64)

    item_unique_ratio = len(set(hist)) / L
    artist_unique_ratio = len(set(artists)) / L
    album_unique_ratio = len(set(albums)) / L

    item_entropy = safe_norm_entropy_from_counts(item_counts)
    artist_entropy = safe_norm_entropy_from_counts(artist_counts)
    duration_entropy = safe_norm_entropy_from_counts(duration_counts)

    diversity_score = float(
        0.35 * item_unique_ratio
        + 0.35 * artist_entropy
        + 0.15 * album_unique_ratio
        + 0.15 * duration_entropy
    )

    return {
        "hist_len": float(L),
        "item_unique_ratio": float(item_unique_ratio),
        "artist_unique_ratio": float(artist_unique_ratio),
        "album_unique_ratio": float(album_unique_ratio),
        "item_entropy": float(item_entropy),
        "artist_entropy": float(artist_entropy),
        "duration_entropy": float(duration_entropy),
        "diversity_score": diversity_score,
    }


def build_eval_analysis_meta(
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    track_pop: pd.Series,
    artist_map: Dict[int, str],
    album_map: Dict[int, str],
    duration_bucket_map: Dict[int, str],
) -> pd.DataFrame:
    rows = []

    pop_values = track_pop.astype(float)
    tail_thr = float(pop_values.quantile(0.20))
    head_thr = float(pop_values.quantile(0.80))

    for idx, u in enumerate(eval_cases.users):
        full_seq = eval_seqs[u]
        hist = full_seq[:-1]
        target = full_seq[-1]

        f = compute_history_diversity_features(
            seq=hist,
            artist_map=artist_map,
            album_map=album_map,
            duration_bucket_map=duration_bucket_map,
        )

        target_pop = float(track_pop.get(target, 0.0))
        if target_pop <= tail_thr:
            target_pop_bucket = "tail"
        elif target_pop >= head_thr:
            target_pop_bucket = "head"
        else:
            target_pop_bucket = "mid"

        last3 = hist[-3:] if len(hist) >= 3 else hist[:]
        repeated_target = int(target in hist)
        target_unseen = int(target not in hist)
        hard_continuation = int(target not in last3)

        rows.append({
            "row_idx": idx,
            "user_id": u,
            "target": target,
            "target_pop": target_pop,
            "target_pop_bucket": target_pop_bucket,
            "repeated_target": repeated_target,
            "target_unseen": target_unseen,
            "hard_continuation": hard_continuation,
            **f,
        })

    meta = pd.DataFrame(rows)

    q_len_33 = meta["hist_len"].quantile(0.33)
    q_len_67 = meta["hist_len"].quantile(0.67)

    def len_bucket_fn(x: float) -> str:
        if x <= q_len_33:
            return "short"
        if x >= q_len_67:
            return "long"
        return "medium"

    meta["len_bucket"] = meta["hist_len"].map(len_bucket_fn)

    q_div_33 = meta["diversity_score"].quantile(0.33)
    q_div_67 = meta["diversity_score"].quantile(0.67)

    def div_bucket_fn(x: float) -> str:
        if x <= q_div_33:
            return "homogeneous"
        if x >= q_div_67:
            return "heterogeneous"
        return "medium"

    meta["div_bucket"] = meta["diversity_score"].map(div_bucket_fn)
    return meta


def build_subset_masks(test_meta: pd.DataFrame) -> Dict[str, np.ndarray]:
    masks: Dict[str, np.ndarray] = {}

    repeated_mask = test_meta["repeated_target"].values.astype(bool)
    nonrepeat_mask = ~repeated_mask

    masks["all"] = np.ones(len(test_meta), dtype=bool)
    masks["all_nonrepeat"] = nonrepeat_mask
    masks["repeated_target"] = repeated_mask

    for bucket in ["short", "medium", "long"]:
        masks[f"len_{bucket}"] = nonrepeat_mask & (test_meta["len_bucket"].values == bucket)

    for bucket in ["homogeneous", "medium", "heterogeneous"]:
        masks[f"div_{bucket}"] = nonrepeat_mask & (test_meta["div_bucket"].values == bucket)

    for bucket in ["head", "mid", "tail"]:
        masks[f"target_pop_{bucket}"] = nonrepeat_mask & (test_meta["target_pop_bucket"].values == bucket)

    masks["target_unseen"] = nonrepeat_mask & test_meta["target_unseen"].values.astype(bool)
    masks["hard_continuation"] = nonrepeat_mask & test_meta["hard_continuation"].values.astype(bool)

    return {k: v for k, v in masks.items() if v.sum() > 0}
    


def metrics_at_k(ranked: np.ndarray, target: np.ndarray, k: int):
    hit = (ranked[:, :k] == target[:, None])
    recall = float(hit.any(axis=1).mean())
    first_pos = hit.argmax(axis=1)
    has_hit = hit.any(axis=1)
    rr = np.zeros(hit.shape[0], dtype=np.float64)
    rr[has_hit] = 1.0 / (first_pos[has_hit] + 1.0)
    mrr = float(rr.mean())
    ndcg = np.zeros(hit.shape[0], dtype=np.float64)
    ndcg[has_hit] = 1.0 / np.log2(first_pos[has_hit] + 2.0)
    ndcg = float(ndcg.mean())
    return recall, ndcg, mrr


def coverage_at_k(ranked: np.ndarray, n_items: int, k: int) -> float:
    vals = ranked[:, :k].reshape(-1)
    vals = vals[vals > 0]
    if len(vals) == 0:
        return 0.0
    return float(len(np.unique(vals)) / n_items)


def compute_metrics_from_ranked(
    ranked: np.ndarray,
    targets: np.ndarray,
    n_items: int,
    k: int,
) -> Dict[str, float]:
    r, n, m = metrics_at_k(ranked, targets, k)
    cov = coverage_at_k(ranked, n_items, k)
    return {
        f"Recall@{k}": float(r),
        f"NDCG@{k}": float(n),
        f"MRR@{k}": float(m),
        f"Coverage@{k}": float(cov),
    }


def summarize_subset_metrics_from_ranked(
    ranked: np.ndarray,
    targets: np.ndarray,
    n_items: int,
    k: int,
    subset_masks: Dict[str, np.ndarray],
    model_name: str,
    seed,
) -> List[Dict[str, object]]:
    rows = []
    for subset_name, mask in subset_masks.items():
        ranked_sub = ranked[mask]
        targets_sub = targets[mask]
        if len(targets_sub) == 0:
            continue

        metrics = compute_metrics_from_ranked(
            ranked=ranked_sub,
            targets=targets_sub,
            n_items=n_items,
            k=k,
        )
        rows.append({
            "model": model_name,
            "seed": seed,
            "subset": subset_name,
            "n_cases": int(mask.sum()),
            **metrics,
        })
    return rows


train_pop_counter = Counter()
for seq in train_seqs.values():
    train_pop_counter.update(seq)

track_pop = pd.Series(
    {item_id: train_pop_counter.get(item_id, 0) for item_id in range(1, n_items + 1)},
    name="track_pop",
    dtype=np.int64,
)

test_analysis_meta = build_eval_analysis_meta(
    eval_cases=test_cases,
    eval_seqs=test_seqs,
    track_pop=track_pop,
    artist_map=itemid_to_artist,
    album_map=itemid_to_album,
    duration_bucket_map=itemid_to_duration_bucket,
)
TEST_SUBSET_MASKS = build_subset_masks(test_analysis_meta)

print("Test analysis meta shape:", test_analysis_meta.shape)
display(test_analysis_meta.head())

Test analysis meta shape: (452067, 18)


,row_idx,user_id,target,target_pop,target_pop_bucket,repeated_target,target_unseen,hard_continuation,hist_len,item_unique_ratio,artist_unique_ratio,album_unique_ratio,item_entropy,artist_entropy,duration_entropy,diversity_score,len_bucket,div_bucket
0,0,222837,2492,2246.0,mid,0,1,1,23.0,1.0,0.956522,0.956522,1.0,0.994881,0.816053,0.964095,short,heterogeneous
1,1,208763,3866,2495.0,mid,0,1,1,55.0,1.0,0.890909,0.945455,1.0,0.988377,0.557954,0.921443,long,medium
2,2,434486,3308,1221.0,mid,0,1,1,12.0,1.0,0.916667,1.000000,1.0,0.988109,0.979574,0.992774,short,heterogeneous
3,3,155443,3558,1244.0,mid,0,1,1,36.0,1.0,0.555556,0.777778,1.0,0.908561,0.873885,0.915746,medium,medium
4,4,160979,662,6108.0,head,0,1,1,51.0,1.0,0.745098,0.901961,1.0,0.964617,0.760479,0.936982,long,medium


In [15]:
print("Available subsets:")
for name, mask in TEST_SUBSET_MASKS.items():
    print(f"{name:>20s}: n={int(mask.sum())}")

Available subsets:
                 all: n=452067
       all_nonrepeat: n=440049
     repeated_target: n=12018
           len_short: n=151955
          len_medium: n=143332
            len_long: n=144762
     div_homogeneous: n=142954
          div_medium: n=150396
   div_heterogeneous: n=146699
     target_pop_head: n=205018
      target_pop_mid: n=190074
     target_pop_tail: n=44957
       target_unseen: n=440049
   hard_continuation: n=440049


In [16]:
pop = Counter()
for seq in train_seqs.values():
    pop.update(seq)

print("TopPop head:", [item for item, _ in pop.most_common(TOPK)])

TopPop head: [275, 1178, 327, 673, 1255, 2058, 273, 277, 1554, 2082, 1111, 1317, 445, 278, 1517, 411, 146, 1288, 1470, 338]


## 9. Training datasets and SSL augmentations

This section defines:
- pairwise next-item training samples,
- CL4SRec-style sequence augmentations,
- SimDCL-style representation perturbation,
- and contrastive objectives used for SSL pretraining.

In [17]:
def sample_negative(n_items: int, forbidden: set[int], rng: random.Random) -> int:
    while True:
        j = rng.randint(1, n_items)
        if j not in forbidden:
            return j


class PairwiseNextItemDataset(Dataset):
    def __init__(
        self,
        train_seqs: Dict[int, List[int]],
        max_len: int,
        n_items: int,
        n_neg: int = 50,
        max_prefix_samples_per_user: Optional[int] = None,
        seed: int = 42,
    ):
        self.instances = []
        rng = random.Random(seed)

        for u, seq in train_seqs.items():
            if len(seq) < 2:
                continue

            positions = list(range(1, len(seq)))
            if max_prefix_samples_per_user is not None and len(positions) > max_prefix_samples_per_user:
                positions = rng.sample(positions, max_prefix_samples_per_user)
                positions.sort()

            forbidden = set(seq)
            playlist_title_tokens = userid_to_playlist_title_tokens.get(
                u, [0] * PLAYLIST_TITLE_MAX_TOKENS
            )

            for t in positions:
                hist = seq[:t]
                pos = seq[t]
                negs = [sample_negative(n_items, forbidden, rng) for _ in range(n_neg)]

                hist_items = right_pad(hist, max_len)
                hist_artist_ids = right_pad([itemid_to_artist_id.get(x, 0) for x in hist], max_len)
                hist_album_ids = right_pad([itemid_to_album_id.get(x, 0) for x in hist], max_len)
                hist_duration_ids = right_pad([itemid_to_duration_id.get(x, 0) for x in hist], max_len)
                hist_track_title_tokens = right_pad(
                    [itemid_to_track_title_tokens.get(x, [0] * ITEM_TITLE_MAX_TOKENS) for x in hist],
                    max_len,
                    pad=[0] * ITEM_TITLE_MAX_TOKENS,
                )

                self.instances.append({
                    "seq_items": hist_items,
                    "seq_artist_ids": hist_artist_ids,
                    "seq_album_ids": hist_album_ids,
                    "seq_duration_ids": hist_duration_ids,
                    "seq_track_title_tokens": hist_track_title_tokens,
                    "playlist_title_tokens": playlist_title_tokens,
                    "pos_item": pos,
                    "pos_artist_id": itemid_to_artist_id.get(pos, 0),
                    "pos_album_id": itemid_to_album_id.get(pos, 0),
                    "pos_duration_id": itemid_to_duration_id.get(pos, 0),
                    "pos_track_title_tokens": itemid_to_track_title_tokens.get(pos, [0] * ITEM_TITLE_MAX_TOKENS),
                    "neg_items": negs,
                    "neg_artist_ids": [itemid_to_artist_id.get(x, 0) for x in negs],
                    "neg_album_ids": [itemid_to_album_id.get(x, 0) for x in negs],
                    "neg_duration_ids": [itemid_to_duration_id.get(x, 0) for x in negs],
                    "neg_track_title_tokens": [itemid_to_track_title_tokens.get(x, [0] * ITEM_TITLE_MAX_TOKENS) for x in negs],
                })

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        row = self.instances[idx]
        return {
            "seq_items": torch.tensor(row["seq_items"], dtype=torch.long),
            "seq_artist_ids": torch.tensor(row["seq_artist_ids"], dtype=torch.long),
            "seq_album_ids": torch.tensor(row["seq_album_ids"], dtype=torch.long),
            "seq_duration_ids": torch.tensor(row["seq_duration_ids"], dtype=torch.long),
            "seq_track_title_tokens": torch.tensor(row["seq_track_title_tokens"], dtype=torch.long),
            "playlist_title_tokens": torch.tensor(row["playlist_title_tokens"], dtype=torch.long),
            "pos_item": torch.tensor(row["pos_item"], dtype=torch.long),
            "pos_artist_id": torch.tensor(row["pos_artist_id"], dtype=torch.long),
            "pos_album_id": torch.tensor(row["pos_album_id"], dtype=torch.long),
            "pos_duration_id": torch.tensor(row["pos_duration_id"], dtype=torch.long),
            "pos_track_title_tokens": torch.tensor(row["pos_track_title_tokens"], dtype=torch.long),
            "neg_items": torch.tensor(row["neg_items"], dtype=torch.long),
            "neg_artist_ids": torch.tensor(row["neg_artist_ids"], dtype=torch.long),
            "neg_album_ids": torch.tensor(row["neg_album_ids"], dtype=torch.long),
            "neg_duration_ids": torch.tensor(row["neg_duration_ids"], dtype=torch.long),
            "neg_track_title_tokens": torch.tensor(row["neg_track_title_tokens"], dtype=torch.long),
        }


class ContrastiveDataset(Dataset):
    def __init__(self, train_seqs: Dict[int, List[int]], max_len: int):
        self.users = list(train_seqs.keys())
        self.train_seqs = train_seqs
        self.max_len = max_len

    def _pack_view(self, u: int, seq: List[int]) -> Dict[str, torch.Tensor]:
        playlist_title_tokens = userid_to_playlist_title_tokens.get(
            u, [0] * PLAYLIST_TITLE_MAX_TOKENS
        )

        seq_items = right_pad(seq, self.max_len)
        seq_artist_ids = right_pad([itemid_to_artist_id.get(x, 0) for x in seq], self.max_len)
        seq_album_ids = right_pad([itemid_to_album_id.get(x, 0) for x in seq], self.max_len)
        seq_duration_ids = right_pad([itemid_to_duration_id.get(x, 0) for x in seq], self.max_len)
        seq_track_title_tokens = right_pad(
            [itemid_to_track_title_tokens.get(x, [0] * ITEM_TITLE_MAX_TOKENS) for x in seq],
            self.max_len,
            pad=[0] * ITEM_TITLE_MAX_TOKENS,
        )

        return {
            "seq_items": torch.tensor(seq_items, dtype=torch.long),
            "seq_artist_ids": torch.tensor(seq_artist_ids, dtype=torch.long),
            "seq_album_ids": torch.tensor(seq_album_ids, dtype=torch.long),
            "seq_duration_ids": torch.tensor(seq_duration_ids, dtype=torch.long),
            "seq_track_title_tokens": torch.tensor(seq_track_title_tokens, dtype=torch.long),
            "playlist_title_tokens": torch.tensor(playlist_title_tokens, dtype=torch.long),
        }

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        seq = self.train_seqs[u]
        v1 = make_view(seq)
        v2 = make_view(seq)
        return self._pack_view(u, v1), self._pack_view(u, v2)


def aug_crop(seq, min_ratio=0.5):
    if len(seq) < 2:
        return seq
    L = len(seq)
    new_len = max(2, int(L * random.uniform(min_ratio, 1.0)))
    start = random.randint(0, L - new_len)
    return seq[start:start + new_len]


def aug_mask(seq, p: float = 0.2):
    if len(seq) <= 1:
        return seq.copy()

    out = []
    for x in seq:
        if random.random() >= p:
            out.append(x)

    if len(out) == 0:
        out = [random.choice(seq)]

    return out


def aug_reorder(seq, window=3):
    out = seq.copy()
    if len(out) <= window:
        random.shuffle(out)
        return out
    start = random.randint(0, len(out) - window)
    sub = out[start:start + window]
    random.shuffle(sub)
    out[start:start + window] = sub
    return out


def make_view(seq):
    s = aug_crop(seq)
    s = aug_reorder(s, window=3)
    s = aug_mask(s, p=0.2)

    if len(s) == 0:
        s = [random.choice(seq)]

    return s


def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)

    logits_12 = (z1 @ z2.t()) / temperature
    logits_21 = (z2 @ z1.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)

    loss_12 = F.cross_entropy(logits_12, labels)
    loss_21 = F.cross_entropy(logits_21, labels)
    return 0.5 * (loss_12 + loss_21)


def simdcl_loss(z: torch.Tensor, noise_std: float, temperature: float):
    z1 = F.normalize(z, dim=-1)
    z2 = F.normalize(z + torch.randn_like(z) * noise_std, dim=-1)
    logits = (z1 @ z2.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    return F.cross_entropy(logits, labels)

## 10. Model architectures

This benchmark includes:
- TopPop
- GRU4Rec
- SASRec
- FMLP-Rec-style efficient baseline
- ComiRec-SA-style SASRec

Transformer-family models use both:
- causal attention masks
- padding masks

The multi-interest model uses:
- target-aware label routing during training
- diversity regularization between interest vectors

In [18]:
def build_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()


def gather_last_nonpad(hidden: torch.Tensor, seq: torch.Tensor) -> torch.Tensor:
    idx = (seq != 0).sum(dim=1) - 1
    idx = idx.clamp_min(0)
    return hidden[torch.arange(hidden.size(0), device=hidden.device), idx]

In [19]:
class MeanTextEncoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, pad_idx: int = 0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        # [*, T]
        x = self.emb(token_ids)
        mask = (token_ids != 0).float().unsqueeze(-1)
        denom = mask.sum(dim=-2).clamp_min(1.0)
        pooled = (x * mask).sum(dim=-2) / denom
        return self.ln(pooled)

In [20]:
class SequenceModelBase(nn.Module):
    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

    def pooled_ssl_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        return self.get_user_repr(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
            playlist_title_tokens,
        )

In [21]:
class MultiInterestBase(nn.Module):
    def get_interest_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

    def score_items_from_features(
        self,
        interest_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

In [22]:
class GRU4Rec(SequenceModelBase):
    def __init__(self, n_items: int, d_model: int, hidden_size: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.proj = nn.Linear(hidden_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(d_model)

    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        x = self.drop(self.item_emb(seq_items))
        h, _ = self.gru(x)
        out = gather_last_nonpad(h, seq_items)
        return self.ln(self.proj(out))

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_repr.unsqueeze(1)).sum(-1)


class MetadataSASRec(SequenceModelBase):
    def __init__(
        self,
        n_items: int,
        n_artists: int,
        n_albums: int,
        n_durations: int,
        n_track_title_tokens: int,
        n_playlist_title_tokens: int,
        max_len: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.artist_emb = nn.Embedding(n_artists, d_model, padding_idx=0)
        self.album_emb = nn.Embedding(n_albums, d_model, padding_idx=0)
        self.duration_emb = nn.Embedding(n_durations, d_model, padding_idx=0)

        self.track_title_encoder = MeanTextEncoder(n_track_title_tokens, d_model, pad_idx=0)
        self.playlist_title_encoder = MeanTextEncoder(n_playlist_title_tokens, d_model, pad_idx=0)

        self.title_gate = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid(),
        )

        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.ln_item = nn.LayerNorm(d_model)
        self.ln_hidden = nn.LayerNorm(d_model)
        self.ln_out = nn.LayerNorm(d_model)

    def encode_item_features(
        self,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_vec = self.item_emb(item_ids)
        artist_vec = self.artist_emb(artist_ids)
        album_vec = self.album_emb(album_ids)
        duration_vec = self.duration_emb(duration_ids)

        orig_shape = track_title_tokens.shape[:-1]
        title_vec = self.track_title_encoder(
            track_title_tokens.reshape(-1, track_title_tokens.shape[-1])
        )
        title_vec = title_vec.reshape(*orig_shape, -1)

        meta_vec = (artist_vec + album_vec + duration_vec + title_vec) / 4.0
        fused = item_vec + 0.3 * meta_vec
        fused = torch.nan_to_num(fused, nan=0.0, posinf=0.0, neginf=0.0)
        return self.ln_item(fused)

    def forward_hidden(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        B, L = seq_items.shape
        pos = torch.arange(L, device=seq_items.device).unsqueeze(0).expand(B, L)

        x = self.encode_item_features(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        x = x + self.pos_emb(pos)
        x = self.drop(x)

        pad_mask = (seq_items == 0)
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        attn_mask = build_causal_mask(L, seq_items.device)

        h = self.encoder(
            x,
            mask=attn_mask,
            src_key_padding_mask=pad_mask,
        )

        h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
        h = h.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return self.ln_hidden(h)

    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        h = self.forward_hidden(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        seq_repr = gather_last_nonpad(h, seq_items)

        playlist_title_repr = self.playlist_title_encoder(playlist_title_tokens)
        playlist_title_repr = torch.nan_to_num(playlist_title_repr, nan=0.0, posinf=0.0, neginf=0.0)

        gate = self.title_gate(torch.cat([seq_repr, playlist_title_repr], dim=-1))
        out = seq_repr + gate * playlist_title_repr
        out = torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
        return self.ln_out(out)

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_repr = self.encode_item_features(
            item_ids,
            artist_ids,
            album_ids,
            duration_ids,
            track_title_tokens,
        )
        scores = (item_repr * user_repr.unsqueeze(1)).sum(dim=-1)
        return torch.nan_to_num(scores, nan=-1e9, posinf=1e9, neginf=-1e9)


class FMLPBlock(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.filter = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        y = self.filter(self.ln1(x))
        x = x + y
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        z = self.ffn(self.ln2(x))
        x = x + z
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return x


class FMLPRec(SequenceModelBase):
    def __init__(self, n_items: int, max_len: int, d_model: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([FMLPBlock(d_model=d_model, dropout=dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)

    def forward_hidden(self, seq_items: torch.Tensor) -> torch.Tensor:
        B, L = seq_items.shape
        pos = torch.arange(L, device=seq_items.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(seq_items) + self.pos_emb(pos)
        x = self.drop(x)
        pad_mask = (seq_items == 0)

        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        for block in self.blocks:
            x = block(x, pad_mask)
        return self.ln(x)

    def get_user_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        h = self.forward_hidden(seq_items)
        return gather_last_nonpad(h, seq_items)

    def score_items_from_features(
        self,
        user_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_repr.unsqueeze(1)).sum(-1)


class MetadataComiRecSA(MultiInterestBase):
    def __init__(
        self,
        n_items: int,
        n_artists: int,
        n_albums: int,
        n_durations: int,
        n_track_title_tokens: int,
        n_playlist_title_tokens: int,
        max_len: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
        n_interests: int,
    ):
        super().__init__()
        self.n_interests = n_interests

        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.artist_emb = nn.Embedding(n_artists, d_model, padding_idx=0)
        self.album_emb = nn.Embedding(n_albums, d_model, padding_idx=0)
        self.duration_emb = nn.Embedding(n_durations, d_model, padding_idx=0)

        self.track_title_encoder = MeanTextEncoder(n_track_title_tokens, d_model, pad_idx=0)
        self.playlist_title_encoder = MeanTextEncoder(n_playlist_title_tokens, d_model, pad_idx=0)

        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.ln_item = nn.LayerNorm(d_model)
        self.ln_hidden = nn.LayerNorm(d_model)

        self.interest_queries = nn.Parameter(torch.randn(n_interests, d_model) * 0.02)
        self.playlist_title_proj = nn.Linear(d_model, d_model)

    def encode_item_features(
        self,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_vec = self.item_emb(item_ids)
        artist_vec = self.artist_emb(artist_ids)
        album_vec = self.album_emb(album_ids)
        duration_vec = self.duration_emb(duration_ids)

        orig_shape = track_title_tokens.shape[:-1]
        title_vec = self.track_title_encoder(
            track_title_tokens.reshape(-1, track_title_tokens.shape[-1])
        )
        title_vec = title_vec.reshape(*orig_shape, -1)

        meta_vec = (artist_vec + album_vec + duration_vec + title_vec) / 4.0
        fused = item_vec + 0.3 * meta_vec
        fused = torch.nan_to_num(fused, nan=0.0, posinf=0.0, neginf=0.0)
        return self.ln_item(fused)

    def forward_hidden(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        B, L = seq_items.shape
        pos = torch.arange(L, device=seq_items.device).unsqueeze(0).expand(B, L)

        x = self.encode_item_features(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        x = x + self.pos_emb(pos)
        x = self.drop(x)

        pad_mask = (seq_items == 0)
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        attn_mask = build_causal_mask(L, seq_items.device)

        h = self.encoder(
            x,
            mask=attn_mask,
            src_key_padding_mask=pad_mask,
        )

        h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
        h = h.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return self.ln_hidden(h)

    def get_interest_repr(
        self,
        seq_items: torch.Tensor,
        seq_artist_ids: torch.Tensor,
        seq_album_ids: torch.Tensor,
        seq_duration_ids: torch.Tensor,
        seq_track_title_tokens: torch.Tensor,
        playlist_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        h = self.forward_hidden(
            seq_items,
            seq_artist_ids,
            seq_album_ids,
            seq_duration_ids,
            seq_track_title_tokens,
        )
        mask = (seq_items != 0)

        B, L, D = h.shape
        q = self.interest_queries.unsqueeze(0).expand(B, self.n_interests, D)

        attn_logits = torch.matmul(q, h.transpose(1, 2)) / math.sqrt(D)
        attn_logits = attn_logits.masked_fill(~mask.unsqueeze(1), -1e9)

        attn = torch.softmax(attn_logits, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0, posinf=0.0, neginf=0.0)

        z = torch.matmul(attn, h)

        playlist_title_repr = self.playlist_title_encoder(playlist_title_tokens)
        playlist_title_repr = torch.nan_to_num(playlist_title_repr, nan=0.0, posinf=0.0, neginf=0.0)

        z = z + self.playlist_title_proj(playlist_title_repr).unsqueeze(1)
        z = torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)

        z = F.normalize(z, dim=-1, eps=1e-8)
        z = torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
        return z

    def score_items_from_features(
        self,
        interest_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_repr = self.encode_item_features(
            item_ids,
            artist_ids,
            album_ids,
            duration_ids,
            track_title_tokens,
        )
        scores = torch.einsum("bkd,bcd->bkc", interest_repr, item_repr)
        scores = scores.max(dim=1).values
        return torch.nan_to_num(scores, nan=-1e9, posinf=1e9, neginf=-1e9)

    def score_items_per_interest_from_features(
        self,
        interest_repr: torch.Tensor,
        item_ids: torch.Tensor,
        artist_ids: torch.Tensor,
        album_ids: torch.Tensor,
        duration_ids: torch.Tensor,
        track_title_tokens: torch.Tensor,
    ) -> torch.Tensor:
        item_repr = self.encode_item_features(
            item_ids,
            artist_ids,
            album_ids,
            duration_ids,
            track_title_tokens,
        )
        scores = torch.einsum("bkd,bcd->bkc", interest_repr, item_repr)
        return torch.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

In [23]:
def predict_toppop_ranked(
    eval_cases: EvalCases,
    train_or_eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    k: int = 20,
) -> np.ndarray:
    item_ids = np.arange(1, n_items + 1, dtype=np.int32)

    global_scores = np.asarray(
        [pop_counter.get(int(i), 0) for i in item_ids],
        dtype=np.int32,
    )

    global_order = np.lexsort((item_ids, -global_scores))
    global_ranked_items = item_ids[global_order]

    ranked = np.zeros((len(eval_cases.users), k), dtype=np.int32)

    for row_idx, u in enumerate(eval_cases.users):
        seen_items = set(train_or_eval_seqs[u][:-1])

        out = []
        for item in global_ranked_items:
            if item not in seen_items:
                out.append(item)
                if len(out) == k:
                    break

        ranked[row_idx, :] = out

    return ranked


def evaluate_toppop_full_ranking(
    eval_cases: EvalCases,
    train_or_eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    k: int = 20,
):
    ranked = predict_toppop_ranked(
        eval_cases=eval_cases,
        train_or_eval_seqs=train_or_eval_seqs,
        pop_counter=pop_counter,
        n_items=n_items,
        k=k,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return metrics[f"Recall@{k}"], metrics[f"NDCG@{k}"], metrics[f"MRR@{k}"], metrics[f"Coverage@{k}"]

In [24]:
def build_eval_batch_tensors(
    batch_users: List[int],
    batch_histories: np.ndarray,
) -> Dict[str, torch.Tensor]:
    return {
        "seq_items": torch.tensor(batch_histories, dtype=torch.long, device=DEVICE),
        "seq_artist_ids": torch.tensor(
            [[itemid_to_artist_id.get(int(x), 0) for x in row] for row in batch_histories],
            dtype=torch.long,
            device=DEVICE,
        ),
        "seq_album_ids": torch.tensor(
            [[itemid_to_album_id.get(int(x), 0) for x in row] for row in batch_histories],
            dtype=torch.long,
            device=DEVICE,
        ),
        "seq_duration_ids": torch.tensor(
            [[itemid_to_duration_id.get(int(x), 0) for x in row] for row in batch_histories],
            dtype=torch.long,
            device=DEVICE,
        ),
        "seq_track_title_tokens": torch.tensor(
            [[itemid_to_track_title_tokens.get(int(x), [0] * ITEM_TITLE_MAX_TOKENS) for x in row] for row in batch_histories],
            dtype=torch.long,
            device=DEVICE,
        ),
        "playlist_title_tokens": torch.tensor(
            [userid_to_playlist_title_tokens.get(int(u), [0] * PLAYLIST_TITLE_MAX_TOKENS) for u in batch_users],
            dtype=torch.long,
            device=DEVICE,
        ),
    }

## 11. Full-ranking, retrieval, and shortlist-reranking utilities

This section contains the shared utilities for:
- chunked full-ranking evaluation
- dense candidate generation
- sparse lexical candidate generation
- hybrid sparse+dense candidate fusion
- shortlist reranking
- IRRT metrics such as candidate recall and reranked ranking quality

In [25]:
@torch.no_grad()
def analyze_interest_usage(model: MultiInterestBase, eval_cases: EvalCases, eval_batch_size: int = 256):
    model.eval()
    all_best_interest, all_interest_entropy = [], []

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        targets = eval_cases.targets[start:end]
        target_item = torch.tensor(targets, dtype=torch.long, device=DEVICE)
        target_artist = torch.tensor([itemid_to_artist_id.get(x, 0) for x in targets], dtype=torch.long, device=DEVICE)
        target_album = torch.tensor([itemid_to_album_id.get(x, 0) for x in targets], dtype=torch.long, device=DEVICE)
        target_duration = torch.tensor([itemid_to_duration_id.get(x, 0) for x in targets], dtype=torch.long, device=DEVICE)
        target_title = torch.tensor(
            [itemid_to_track_title_tokens.get(x, [0] * ITEM_TITLE_MAX_TOKENS) for x in targets],
            dtype=torch.long,
            device=DEVICE,
        )

        z = model.get_interest_repr(
            batch["seq_items"],
            batch["seq_artist_ids"],
            batch["seq_album_ids"],
            batch["seq_duration_ids"],
            batch["seq_track_title_tokens"],
            batch["playlist_title_tokens"],
        )

        interest_scores = model.score_items_per_interest_from_features(
            z,
            target_item.unsqueeze(1),
            target_artist.unsqueeze(1),
            target_album.unsqueeze(1),
            target_duration.unsqueeze(1),
            target_title.unsqueeze(1),
        ).squeeze(-1)

        best_interest = interest_scores.argmax(dim=1).cpu().numpy()
        probs = torch.softmax(interest_scores, dim=1)
        entropy = (-(probs * probs.clamp_min(1e-9).log()).sum(dim=1)).cpu().numpy()

        all_best_interest.extend(best_interest.tolist())
        all_interest_entropy.extend(entropy.tolist())

    counts = np.bincount(np.asarray(all_best_interest), minlength=model.n_interests).astype(np.float64)
    counts = counts / counts.sum()

    return {
        "interest_usage_entropy": float(-(counts * np.log(counts + 1e-12)).sum()),
        "interest_usage_max_share": float(counts.max()),
        "interest_target_score_entropy_mean": float(np.mean(all_interest_entropy)),
    }


class EarlyStopper:
    def __init__(self, patience: int = 5, min_delta: float = 1e-4, mode: str = "max"):
        assert mode in ("max", "min")
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.bad_epochs = 0
        self.best_state = None

    def _is_improvement(self, value: float) -> bool:
        if self.best is None:
            return True
        if self.mode == "max":
            return value > self.best + self.min_delta
        return value < self.best - self.min_delta

    def step(self, value: float, model: nn.Module) -> bool:
        if self._is_improvement(value):
            self.best = value
            self.bad_epochs = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    return -torch.log(torch.sigmoid(pos_scores.unsqueeze(1) - neg_scores) + 1e-9).mean()


def diversity_regularization(z: torch.Tensor) -> torch.Tensor:
    z = F.normalize(z, dim=-1)
    sim = torch.matmul(z, z.transpose(1, 2))
    K = sim.size(1)
    eye = torch.eye(K, device=sim.device).unsqueeze(0)
    off_diag = sim * (1.0 - eye)
    return (off_diag ** 2).mean()


def label_aware_comirec_loss(
    model: MetadataComiRecSA,
    batch: Dict[str, torch.Tensor],
    label_tau: float,
    diversity_reg_weight: float,
    use_label_aware: bool = True,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    z = model.get_interest_repr(
        batch["seq_items"],
        batch["seq_artist_ids"],
        batch["seq_album_ids"],
        batch["seq_duration_ids"],
        batch["seq_track_title_tokens"],
        batch["playlist_title_tokens"],
    )

    pos_scores_per_interest = model.score_items_per_interest_from_features(
        z,
        batch["pos_item"].unsqueeze(1),
        batch["pos_artist_id"].unsqueeze(1),
        batch["pos_album_id"].unsqueeze(1),
        batch["pos_duration_id"].unsqueeze(1),
        batch["pos_track_title_tokens"].unsqueeze(1),
    ).squeeze(-1)

    neg_scores_per_interest = model.score_items_per_interest_from_features(
        z,
        batch["neg_items"],
        batch["neg_artist_ids"],
        batch["neg_album_ids"],
        batch["neg_duration_ids"],
        batch["neg_track_title_tokens"],
    )

    pos_scores_per_interest = torch.nan_to_num(pos_scores_per_interest, nan=0.0, posinf=0.0, neginf=0.0)
    neg_scores_per_interest = torch.nan_to_num(neg_scores_per_interest, nan=0.0, posinf=0.0, neginf=0.0)

    if use_label_aware:
        routing_weights = torch.softmax(pos_scores_per_interest / max(label_tau, 1e-6), dim=1)
        routing_weights = torch.nan_to_num(routing_weights, nan=0.0, posinf=0.0, neginf=0.0)
        pos_scores = (routing_weights * pos_scores_per_interest).sum(dim=1)
        neg_scores = (routing_weights.unsqueeze(-1) * neg_scores_per_interest).sum(dim=1)
    else:
        pos_scores = pos_scores_per_interest.max(dim=1).values
        neg_scores = neg_scores_per_interest.max(dim=1).values

    pos_scores = torch.nan_to_num(pos_scores, nan=0.0, posinf=0.0, neginf=0.0)
    neg_scores = torch.nan_to_num(neg_scores, nan=0.0, posinf=0.0, neginf=0.0)

    rec_loss = bpr_loss(pos_scores, neg_scores)
    div_loss = diversity_regularization(z)

    rec_loss = torch.nan_to_num(rec_loss, nan=0.0, posinf=1e3, neginf=1e3)
    div_loss = torch.nan_to_num(div_loss, nan=0.0, posinf=1e3, neginf=1e3)

    total_loss = rec_loss + diversity_reg_weight * div_loss

    aux = {
        "rec_loss": float(rec_loss.detach().item()),
        "div_loss": float(div_loss.detach().item()),
    }
    return total_loss, aux


def make_pairwise_loader(train_seqs, seed: int):
    ds = PairwiseNextItemDataset(
        train_seqs=train_seqs,
        max_len=MAX_SEQ_LEN,
        n_items=n_items,
        n_neg=50,
        max_prefix_samples_per_user=20,
        seed=seed,
    )
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
    return ds, dl


def build_all_item_feature_tensors_cpu() -> Dict[str, torch.Tensor]:
    return {
        "item_ids": torch.arange(1, n_items + 1, dtype=torch.long),
        "artist_ids": torch.tensor([itemid_to_artist_id.get(i, 0) for i in range(1, n_items + 1)], dtype=torch.long),
        "album_ids": torch.tensor([itemid_to_album_id.get(i, 0) for i in range(1, n_items + 1)], dtype=torch.long),
        "duration_ids": torch.tensor([itemid_to_duration_id.get(i, 0) for i in range(1, n_items + 1)], dtype=torch.long),
        "track_title_tokens": torch.tensor(
            [itemid_to_track_title_tokens.get(i, [0] * ITEM_TITLE_MAX_TOKENS) for i in range(1, n_items + 1)],
            dtype=torch.long,
        ),
    }


ALL_ITEM_FEATURES_CPU = build_all_item_feature_tensors_cpu()


def ranked_array_to_dict(ranked: np.ndarray, users: List[int]) -> Dict[int, List[int]]:
    out: Dict[int, List[int]] = {}
    for u, row in zip(users, ranked):
        out[u] = [int(x) for x in row.tolist() if int(x) > 0]
    return out


def ranked_dict_to_array(
    ranked_dict: Dict[int, List[int]],
    users: List[int],
    k: int,
) -> np.ndarray:
    arr = np.zeros((len(users), k), dtype=np.int64)
    for i, u in enumerate(users):
        vals = ranked_dict.get(u, [])[:k]
        if vals:
            arr[i, :len(vals)] = np.asarray(vals, dtype=np.int64)
    return arr


def compute_metrics_from_ranked_dict(
    ranked_dict: Dict[int, List[int]],
    eval_cases: EvalCases,
    n_items: int,
    k: int,
) -> Dict[str, float]:
    arr = ranked_dict_to_array(ranked_dict, eval_cases.users, k)
    return compute_metrics_from_ranked(arr, eval_cases.targets, n_items, k)


@torch.no_grad()
def predict_repr_model_ranked(
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> np.ndarray:
    model.eval()
    ranked_all = []

    all_item_ids_cpu = ALL_ITEM_FEATURES_CPU["item_ids"]
    all_artist_ids_cpu = ALL_ITEM_FEATURES_CPU["artist_ids"]
    all_album_ids_cpu = ALL_ITEM_FEATURES_CPU["album_ids"]
    all_duration_ids_cpu = ALL_ITEM_FEATURES_CPU["duration_ids"]
    all_track_title_tokens_cpu = ALL_ITEM_FEATURES_CPU["track_title_tokens"]

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        user_repr = model.get_user_repr(
            batch["seq_items"],
            batch["seq_artist_ids"],
            batch["seq_album_ids"],
            batch["seq_duration_ids"],
            batch["seq_track_title_tokens"],
            batch["playlist_title_tokens"],
        )
        batch_size = batch["seq_items"].size(0)

        seen_sets = [set(eval_seqs[u][:-1]) for u in batch_users]

        batch_top_scores = torch.full((batch_size, k), -1e9, dtype=torch.float32, device=DEVICE)
        batch_top_items = torch.zeros((batch_size, k), dtype=torch.long, device=DEVICE)

        for item_start in range(0, n_items, item_chunk_size):
            item_end = min(item_start + item_chunk_size, n_items)

            chunk_item_ids = all_item_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_artist_ids = all_artist_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_album_ids = all_album_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_duration_ids = all_duration_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_track_title_tokens = all_track_title_tokens_cpu[item_start:item_end].to(DEVICE)

            scores_chunk = model.score_items_from_features(
                user_repr,
                chunk_item_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_artist_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_album_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_duration_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_track_title_tokens.unsqueeze(0).expand(batch_size, -1, -1),
            )

            for i, seen_items in enumerate(seen_sets):
                local_seen = [item - item_start - 1 for item in seen_items if item_start < item <= item_end]
                if local_seen:
                    scores_chunk[i, local_seen] = -1e9

            chunk_k = min(k, scores_chunk.size(1))
            chunk_top_scores, chunk_top_idx = torch.topk(scores_chunk, k=chunk_k, dim=1)
            chunk_top_items = chunk_item_ids[chunk_top_idx]

            merged_scores = torch.cat([batch_top_scores, chunk_top_scores], dim=1)
            merged_items = torch.cat([batch_top_items, chunk_top_items], dim=1)

            new_top_scores, new_top_pos = torch.topk(merged_scores, k=k, dim=1)
            new_top_items = torch.gather(merged_items, 1, new_top_pos)

            batch_top_scores = new_top_scores
            batch_top_items = new_top_items

        ranked_all.append(batch_top_items.cpu().numpy())

    return np.vstack(ranked_all)


@torch.no_grad()
def evaluate_repr_model_full_ranking(
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
):
    ranked = predict_repr_model_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return metrics[f"Recall@{k}"], metrics[f"NDCG@{k}"], metrics[f"MRR@{k}"], metrics[f"Coverage@{k}"]


@torch.no_grad()
def predict_multi_interest_ranked(
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> np.ndarray:
    model.eval()
    ranked_all = []

    all_item_ids_cpu = ALL_ITEM_FEATURES_CPU["item_ids"]
    all_artist_ids_cpu = ALL_ITEM_FEATURES_CPU["artist_ids"]
    all_album_ids_cpu = ALL_ITEM_FEATURES_CPU["album_ids"]
    all_duration_ids_cpu = ALL_ITEM_FEATURES_CPU["duration_ids"]
    all_track_title_tokens_cpu = ALL_ITEM_FEATURES_CPU["track_title_tokens"]

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        z = model.get_interest_repr(
            batch["seq_items"],
            batch["seq_artist_ids"],
            batch["seq_album_ids"],
            batch["seq_duration_ids"],
            batch["seq_track_title_tokens"],
            batch["playlist_title_tokens"],
        )
        batch_size = batch["seq_items"].size(0)

        seen_sets = [set(eval_seqs[u][:-1]) for u in batch_users]

        batch_top_scores = torch.full((batch_size, k), -1e9, dtype=torch.float32, device=DEVICE)
        batch_top_items = torch.zeros((batch_size, k), dtype=torch.long, device=DEVICE)

        for item_start in range(0, n_items, item_chunk_size):
            item_end = min(item_start + item_chunk_size, n_items)

            chunk_item_ids = all_item_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_artist_ids = all_artist_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_album_ids = all_album_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_duration_ids = all_duration_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_track_title_tokens = all_track_title_tokens_cpu[item_start:item_end].to(DEVICE)

            scores_chunk = model.score_items_from_features(
                z,
                chunk_item_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_artist_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_album_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_duration_ids.unsqueeze(0).expand(batch_size, -1),
                chunk_track_title_tokens.unsqueeze(0).expand(batch_size, -1, -1),
            )

            for i, seen_items in enumerate(seen_sets):
                local_seen = [item - item_start - 1 for item in seen_items if item_start < item <= item_end]
                if local_seen:
                    scores_chunk[i, local_seen] = -1e9

            chunk_k = min(k, scores_chunk.size(1))
            chunk_top_scores, chunk_top_idx = torch.topk(scores_chunk, k=chunk_k, dim=1)
            chunk_top_items = chunk_item_ids[chunk_top_idx]

            merged_scores = torch.cat([batch_top_scores, chunk_top_scores], dim=1)
            merged_items = torch.cat([batch_top_items, chunk_top_items], dim=1)

            new_top_scores, new_top_pos = torch.topk(merged_scores, k=k, dim=1)
            new_top_items = torch.gather(merged_items, 1, new_top_pos)

            batch_top_scores = new_top_scores
            batch_top_items = new_top_items

        ranked_all.append(batch_top_items.cpu().numpy())

    return np.vstack(ranked_all)


@torch.no_grad()
def evaluate_multi_interest_full_ranking(
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
):
    ranked = predict_multi_interest_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return metrics[f"Recall@{k}"], metrics[f"NDCG@{k}"], metrics[f"MRR@{k}"], metrics[f"Coverage@{k}"]


def build_sparse_inverted_index(
    item_sparse_vecs: Dict[int, Dict[str, float]]
) -> Dict[str, List[Tuple[int, float]]]:
    inv: Dict[str, List[Tuple[int, float]]] = {}
    for item_id, vec in item_sparse_vecs.items():
        for tok, weight in vec.items():
            inv.setdefault(tok, []).append((item_id, weight))
    return inv


SPARSE_INVERTED_INDEX = build_sparse_inverted_index(itemid_to_sparse_vec)


def build_sparse_candidate_dict(
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    top_m: int,
    title_weight: float,
    history_weight: float,
    history_last_n: int,
) -> Dict[int, List[int]]:
    out: Dict[int, List[int]] = {}

    for u in eval_cases.users:
        history_items = eval_seqs[u][:-1]
        seen_items = set(history_items)

        query_vec = build_sparse_query_from_user_context(
            user_id=u,
            history_items=history_items,
            title_weight=title_weight,
            history_weight=history_weight,
            history_last_n=history_last_n,
        )

        score_accum: Dict[int, float] = {}
        for tok, q_weight in query_vec.items():
            postings = SPARSE_INVERTED_INDEX.get(tok, [])
            for item_id, d_weight in postings:
                if item_id in seen_items:
                    continue
                score_accum[item_id] = score_accum.get(item_id, 0.0) + q_weight * d_weight

        ranked = sorted(score_accum.items(), key=lambda x: (-x[1], x[0]))
        out[u] = [item_id for item_id, _ in ranked[:top_m]]

    return out


def build_sparse_query_from_user_context(
    user_id: int,
    history_items: List[int],
    title_weight: float = 2.0,
    history_weight: float = 1.0,
    history_last_n: int = 20,
) -> Dict[str, float]:
    query: Counter = Counter()

    title_text = userid_to_playlist_title_text.get(user_id, "")
    title_tokens = tokenize_text_simple(title_text)
    for tok in title_tokens:
        query[tok] += title_weight

    history_slice = history_items[-history_last_n:]
    for item_id in history_slice:
        text = itemid_to_lexical_text.get(item_id, "")
        toks = tokenize_text_simple(text)
        for tok in toks:
            query[tok] += history_weight

    out: Dict[str, float] = {}
    for tok, tf in query.items():
        df_val = LEXICAL_DF.get(tok, 0)
        idf = math.log((1.0 + LEXICAL_N_DOCS) / (1.0 + df_val)) + 1.0
        out[tok] = float(tf) * idf

    return out


def build_dense_candidate_dict(
    model: nn.Module,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    top_m: int,
    eval_batch_size: int,
    item_chunk_size: int,
) -> Dict[int, List[int]]:
    if isinstance(model, MultiInterestBase):
        ranked = predict_multi_interest_ranked(
            model=model,
            eval_cases=eval_cases,
            eval_seqs=eval_seqs,
            n_items=n_items,
            k=top_m,
            eval_batch_size=eval_batch_size,
            item_chunk_size=item_chunk_size,
        )
    else:
        ranked = predict_repr_model_ranked(
            model=model,
            eval_cases=eval_cases,
            eval_seqs=eval_seqs,
            n_items=n_items,
            k=top_m,
            eval_batch_size=eval_batch_size,
            item_chunk_size=item_chunk_size,
        )
    return ranked_array_to_dict(ranked, eval_cases.users)


def hybrid_fuse_candidates(
    dense_candidates: List[int],
    sparse_candidates: List[int],
    dense_weight: float = 1.0,
    sparse_weight: float = 1.0,
    top_m: int = 200,
) -> List[int]:
    score: Dict[int, float] = {}

    for rank, item in enumerate(dense_candidates):
        score[item] = score.get(item, 0.0) + dense_weight / (rank + 1)

    for rank, item in enumerate(sparse_candidates):
        score[item] = score.get(item, 0.0) + sparse_weight / (rank + 1)

    fused = sorted(score.items(), key=lambda x: (-x[1], x[0]))
    return [item for item, _ in fused[:top_m]]


def build_hybrid_candidate_dict(
    dense_dict: Dict[int, List[int]],
    sparse_dict: Dict[int, List[int]],
    dense_weight: float,
    sparse_weight: float,
    top_m: int,
) -> Dict[int, List[int]]:
    out: Dict[int, List[int]] = {}
    users = sorted(set(dense_dict.keys()) | set(sparse_dict.keys()))

    for u in users:
        out[u] = hybrid_fuse_candidates(
            dense_dict.get(u, []),
            sparse_dict.get(u, []),
            dense_weight=dense_weight,
            sparse_weight=sparse_weight,
            top_m=top_m,
        )

    return out


def candidate_recall_at_m(
    candidate_dict: Dict[int, List[int]],
    eval_cases: EvalCases,
    m: int,
) -> float:
    hits = 0
    total = 0

    for u, target in zip(eval_cases.users, eval_cases.targets):
        total += 1
        candidates = candidate_dict.get(u, [])[:m]
        hits += int(int(target) in candidates)

    return float(hits / max(1, total))

In [26]:
# %% [markdown]
# ## 10.1 Debug sanity checks for metadata models

# %%
@torch.no_grad()
def debug_model_forward_once(model: nn.Module, train_seqs: Dict[int, List[int]], n_batches: int = 1):
    model.eval()
    _, dl = make_pairwise_loader(train_seqs, seed=42)

    for batch_idx, batch in enumerate(dl):
        if batch_idx >= n_batches:
            break

        for kk in batch:
            batch[kk] = batch[kk].to(DEVICE)

        print("=" * 80)
        print(f"Batch {batch_idx}")
        print("seq_items shape:", tuple(batch["seq_items"].shape))
        print("pos_item shape:", tuple(batch["pos_item"].shape))
        print("neg_items shape:", tuple(batch["neg_items"].shape))

        if isinstance(model, MultiInterestBase):
            z = model.get_interest_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )
            print("interest repr shape:", tuple(z.shape))
            print("interest repr mean abs:", float(z.abs().mean().item()))
            print("interest repr std:", float(z.std().item()))

            pos_scores = model.score_items_per_interest_from_features(
                z,
                batch["pos_item"].unsqueeze(1),
                batch["pos_artist_id"].unsqueeze(1),
                batch["pos_album_id"].unsqueeze(1),
                batch["pos_duration_id"].unsqueeze(1),
                batch["pos_track_title_tokens"].unsqueeze(1),
            ).squeeze(-1)

            neg_scores = model.score_items_per_interest_from_features(
                z,
                batch["neg_items"],
                batch["neg_artist_ids"],
                batch["neg_album_ids"],
                batch["neg_duration_ids"],
                batch["neg_track_title_tokens"],
            )

            print("pos_scores_per_interest shape:", tuple(pos_scores.shape))
            print("neg_scores_per_interest shape:", tuple(neg_scores.shape))
            print("pos_scores mean:", float(pos_scores.mean().item()))
            print("pos_scores std:", float(pos_scores.std().item()))
            print("neg_scores mean:", float(neg_scores.mean().item()))
            print("neg_scores std:", float(neg_scores.std().item()))
        else:
            u = model.get_user_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )
            print("user repr shape:", tuple(u.shape))
            print("user repr mean abs:", float(u.abs().mean().item()))
            print("user repr std:", float(u.std().item()))

            pos_scores = model.score_items_from_features(
                u,
                batch["pos_item"].unsqueeze(1),
                batch["pos_artist_id"].unsqueeze(1),
                batch["pos_album_id"].unsqueeze(1),
                batch["pos_duration_id"].unsqueeze(1),
                batch["pos_track_title_tokens"].unsqueeze(1),
            ).squeeze(1)

            neg_scores = model.score_items_from_features(
                u,
                batch["neg_items"],
                batch["neg_artist_ids"],
                batch["neg_album_ids"],
                batch["neg_duration_ids"],
                batch["neg_track_title_tokens"],
            )

            print("pos_scores shape:", tuple(pos_scores.shape))
            print("neg_scores shape:", tuple(neg_scores.shape))
            print("pos_scores mean:", float(pos_scores.mean().item()))
            print("pos_scores std:", float(pos_scores.std().item()))
            print("neg_scores mean:", float(neg_scores.mean().item()))
            print("neg_scores std:", float(neg_scores.std().item()))
            print("mean(pos - neg):", float((pos_scores.unsqueeze(1) - neg_scores).mean().item()))

In [27]:
# %%
def debug_one_training_step(model: nn.Module, train_seqs: Dict[int, List[int]], lr: float = 1e-3):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    _, dl = make_pairwise_loader(train_seqs, seed=42)
    batch = next(iter(dl))

    for kk in batch:
        batch[kk] = batch[kk].to(DEVICE)

    if isinstance(model, MultiInterestBase):
        loss, aux = label_aware_comirec_loss(
            model=model,
            batch=batch,
            label_tau=COMIREC_LABEL_TAU,
            diversity_reg_weight=COMIREC_DIVERSITY_REG,
            use_label_aware=True,
        )
        print("loss:", float(loss.item()), "| aux:", aux)
    else:
        u = model.get_user_repr(
            batch["seq_items"],
            batch["seq_artist_ids"],
            batch["seq_album_ids"],
            batch["seq_duration_ids"],
            batch["seq_track_title_tokens"],
            batch["playlist_title_tokens"],
        )

        pos_scores = model.score_items_from_features(
            u,
            batch["pos_item"].unsqueeze(1),
            batch["pos_artist_id"].unsqueeze(1),
            batch["pos_album_id"].unsqueeze(1),
            batch["pos_duration_id"].unsqueeze(1),
            batch["pos_track_title_tokens"].unsqueeze(1),
        ).squeeze(1)

        neg_scores = model.score_items_from_features(
            u,
            batch["neg_items"],
            batch["neg_artist_ids"],
            batch["neg_album_ids"],
            batch["neg_duration_ids"],
            batch["neg_track_title_tokens"],
        )

        loss = bpr_loss(pos_scores, neg_scores)
        print("loss:", float(loss.item()))
        print("mean(pos - neg):", float((pos_scores.unsqueeze(1) - neg_scores).mean().item()))

    opt.zero_grad()
    loss.backward()

    grad_stats = []
    for name, p in model.named_parameters():
        if p.grad is not None:
            grad_norm = float(p.grad.norm().item())
            grad_stats.append((name, grad_norm))

    grad_stats = sorted(grad_stats, key=lambda x: -x[1])
    print("\nTop gradient norms:")
    for name, gn in grad_stats[:15]:
        print(f"{name:60s} {gn:.6f}")

    nonzero_grads = sum(1 for _, gn in grad_stats if gn > 0)
    print("\nParameters with non-zero gradients:", nonzero_grads, "/", len(grad_stats))

In [29]:
def sparse_dot_score(query_vec: Dict[str, float], doc_vec: Dict[str, float]) -> float:
    if len(query_vec) > len(doc_vec):
        query_vec, doc_vec = doc_vec, query_vec
    return float(sum(v * doc_vec.get(k, 0.0) for k, v in query_vec.items()))


@torch.no_grad()
def rerank_candidate_dict_with_model(
    model: nn.Module,
    eval_cases: EvalCases,
    candidate_dict: Dict[int, List[int]],
    k: int,
    eval_batch_size: int,
) -> Dict[int, List[int]]:
    model.eval()
    out: Dict[int, List[int]] = {}

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        batch_users = eval_cases.users[start:end]
        batch_histories = eval_cases.histories[start:end]

        batch = build_eval_batch_tensors(batch_users, batch_histories)

        candidate_lists = [candidate_dict.get(u, []) for u in batch_users]
        max_cands = max((len(x) for x in candidate_lists), default=0)

        if max_cands == 0:
            for u in batch_users:
                out[u] = []
            continue

        cand_item_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_artist_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_album_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_duration_ids = torch.zeros((len(batch_users), max_cands), dtype=torch.long, device=DEVICE)
        cand_track_title_tokens = torch.zeros(
            (len(batch_users), max_cands, ITEM_TITLE_MAX_TOKENS),
            dtype=torch.long,
            device=DEVICE,
        )
        cand_mask = torch.zeros((len(batch_users), max_cands), dtype=torch.bool, device=DEVICE)

        for i, cand_list in enumerate(candidate_lists):
            for j, item_id in enumerate(cand_list):
                cand_item_ids[i, j] = item_id
                cand_artist_ids[i, j] = itemid_to_artist_id.get(item_id, 0)
                cand_album_ids[i, j] = itemid_to_album_id.get(item_id, 0)
                cand_duration_ids[i, j] = itemid_to_duration_id.get(item_id, 0)
                cand_track_title_tokens[i, j] = torch.tensor(
                    itemid_to_track_title_tokens.get(item_id, [0] * ITEM_TITLE_MAX_TOKENS),
                    dtype=torch.long,
                    device=DEVICE,
                )
                cand_mask[i, j] = True

        if isinstance(model, MultiInterestBase):
            interest_repr = model.get_interest_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )
            scores = model.score_items_from_features(
                interest_repr,
                cand_item_ids,
                cand_artist_ids,
                cand_album_ids,
                cand_duration_ids,
                cand_track_title_tokens,
            )
        else:
            user_repr = model.get_user_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )
            scores = model.score_items_from_features(
                user_repr,
                cand_item_ids,
                cand_artist_ids,
                cand_album_ids,
                cand_duration_ids,
                cand_track_title_tokens,
            )

        scores = scores.masked_fill(~cand_mask, -1e9)

        topk = min(k, max_cands)
        _, top_idx = torch.topk(scores, k=topk, dim=1)
        top_items = torch.gather(cand_item_ids, 1, top_idx)

        for i, u in enumerate(batch_users):
            out[u] = [int(x) for x in top_items[i].tolist() if int(x) > 0][:k]

    return out

## 12. Evaluation helpers and subset reporting

This section contains:
- subset-based evaluation helpers,
- TopPop full-ranking evaluation,
- and utilities for building per-model subset result tables.

In [30]:
def build_test_subset_rows_for_toppop(
    model_name: str,
    seed,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
) -> List[Dict[str, object]]:
    ranked = predict_toppop_ranked(
        eval_cases=eval_cases,
        train_or_eval_seqs=eval_seqs,
        pop_counter=pop_counter,
        n_items=n_items,
        k=k,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )


def build_test_subset_rows_for_repr_model(
    model_name: str,
    seed,
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> List[Dict[str, object]]:
    ranked = predict_repr_model_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )


def build_test_subset_rows_for_multi_interest_model(
    model_name: str,
    seed,
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> List[Dict[str, object]]:
    ranked = predict_multi_interest_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )

In [31]:
top_r, top_n, top_m, top_cov = evaluate_toppop_full_ranking(
    test_cases,
    test_seqs,
    pop,
    n_items,
    TOPK,
)
print(f"TopPop TEST: R@{TOPK}={top_r:.4f} N@{TOPK}={top_n:.4f} MRR@{TOPK}={top_m:.4f} Coverage@{TOPK}={top_cov:.4f}")

TopPop TEST: R@20=0.0185 N@20=0.0075 MRR@20=0.0045 Coverage@20=0.0060


## 13. Training loops for benchmark models

This section contains the training procedures for:
- GRU4Rec
- SASRec
- SASRec + CL4SRec-style SSL
- SASRec + SimDCL-style SSL
- FMLP-Rec-style efficient baseline
- ComiRec-SA-style SASRec
- ComiRec-SA-style SASRec with label-aware routing and diversity regularization

All models use the same validation protocol and early stopping logic.

In [32]:
def train_gru4rec(train_seqs, valid_cases, test_cases, seed: int):
    set_all_seeds(seed)
    model = GRU4Rec(
        n_items=n_items,
        d_model=D_MODEL,
        hidden_size=GRU_HIDDEN,
        n_layers=1,
        dropout=DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for batch in dl:
            for kk in batch:
                batch[kk] = batch[kk].to(DEVICE)

            u = model.get_user_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )

            pos_scores = model.score_items_from_features(
                u,
                batch["pos_item"].unsqueeze(1),
                batch["pos_artist_id"].unsqueeze(1),
                batch["pos_album_id"].unsqueeze(1),
                batch["pos_duration_id"].unsqueeze(1),
                batch["pos_track_title_tokens"].unsqueeze(1),
            ).squeeze(1)

            neg_scores = model.score_items_from_features(
                u,
                batch["neg_items"],
                batch["neg_artist_ids"],
                batch["neg_album_ids"],
                batch["neg_duration_ids"],
                batch["neg_track_title_tokens"],
            )

            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(f"[GRU4Rec][seed={seed}] epoch {ep+1}/{EPOCHS}: loss={total/max(1, len(dl)):.4f} | valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}")

        if stopper.step(n, model):
            print(f"[GRU4Rec][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name="GRU4Rec",
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows


def train_metadata_sasrec(train_seqs, valid_cases, test_cases, seed: int, ssl_mode: Optional[str] = None):
    set_all_seeds(seed)
    model = MetadataSASRec(
        n_items=n_items,
        n_artists=N_ARTISTS,
        n_albums=N_ALBUMS,
        n_durations=N_DURATIONS,
        n_track_title_tokens=N_TRACK_TITLE_TOKENS,
        n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    if ssl_mode is not None:
        ds_ssl = ContrastiveDataset(train_seqs, MAX_SEQ_LEN)
        dl_ssl = DataLoader(ds_ssl, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)

        for ep in range(SSL_EPOCHS):
            model.train()
            total = 0.0
            for v1, v2 in dl_ssl:
                for kk in v1:
                    v1[kk] = v1[kk].to(DEVICE)
                    v2[kk] = v2[kk].to(DEVICE)

                if ssl_mode == "cl4srec":
                    z1 = model.pooled_ssl_repr(**v1)
                    z2 = model.pooled_ssl_repr(**v2)
                    loss = info_nce_loss(z1, z2, temperature=SSL_TAU)
                elif ssl_mode == "simdcl":
                    z = model.pooled_ssl_repr(**v1)
                    loss = simdcl_loss(z, noise_std=SIMDCL_NOISE_STD, temperature=SSL_TAU)
                else:
                    raise ValueError(ssl_mode)

                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                total += float(loss.item())

            print(f"[MetadataSASRec SSL-{ssl_mode}] epoch {ep+1}/{SSL_EPOCHS}: loss={total/max(1, len(dl_ssl)):.4f}")

    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    tag = {
        None: "MetadataSASRec",
        "cl4srec": "MetadataSASRec+CL4SRec",
        "simdcl": "MetadataSASRec+SimDCL",
    }[ssl_mode]

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for batch in dl:
            for kk in batch:
                batch[kk] = batch[kk].to(DEVICE)

            u = model.get_user_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )

            pos_scores = model.score_items_from_features(
                u,
                batch["pos_item"].unsqueeze(1),
                batch["pos_artist_id"].unsqueeze(1),
                batch["pos_album_id"].unsqueeze(1),
                batch["pos_duration_id"].unsqueeze(1),
                batch["pos_track_title_tokens"].unsqueeze(1),
            ).squeeze(1)

            neg_scores = model.score_items_from_features(
                u,
                batch["neg_items"],
                batch["neg_artist_ids"],
                batch["neg_album_ids"],
                batch["neg_duration_ids"],
                batch["neg_track_title_tokens"],
            )

            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(f"[{tag}][seed={seed}] epoch {ep+1}/{EPOCHS}: loss={total/max(1, len(dl)):.4f} | valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}")

        if stopper.step(n, model):
            print(f"[{tag}][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name=tag,
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows


def train_fmlprec(train_seqs, valid_cases, test_cases, seed: int):
    set_all_seeds(seed)
    model = FMLPRec(
        n_items=n_items,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_layers=FMLP_N_LAYERS,
        dropout=FMLP_DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for batch in dl:
            for kk in batch:
                batch[kk] = batch[kk].to(DEVICE)

            u = model.get_user_repr(
                batch["seq_items"],
                batch["seq_artist_ids"],
                batch["seq_album_ids"],
                batch["seq_duration_ids"],
                batch["seq_track_title_tokens"],
                batch["playlist_title_tokens"],
            )

            pos_scores = model.score_items_from_features(
                u,
                batch["pos_item"].unsqueeze(1),
                batch["pos_artist_id"].unsqueeze(1),
                batch["pos_album_id"].unsqueeze(1),
                batch["pos_duration_id"].unsqueeze(1),
                batch["pos_track_title_tokens"].unsqueeze(1),
            ).squeeze(1)

            neg_scores = model.score_items_from_features(
                u,
                batch["neg_items"],
                batch["neg_artist_ids"],
                batch["neg_album_ids"],
                batch["neg_duration_ids"],
                batch["neg_track_title_tokens"],
            )

            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(f"[FMLP-Rec][seed={seed}] epoch {ep+1}/{EPOCHS}: loss={total/max(1, len(dl)):.4f} | valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}")

        if stopper.step(n, model):
            print(f"[FMLP-Rec][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name="FMLP-Rec",
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows


def train_comirec_sa_style(train_seqs, valid_cases, test_cases, seed: int, use_label_aware: bool):
    set_all_seeds(seed)
    model = MetadataComiRecSA(
        n_items=n_items,
        n_artists=N_ARTISTS,
        n_albums=N_ALBUMS,
        n_durations=N_DURATIONS,
        n_track_title_tokens=N_TRACK_TITLE_TOKENS,
        n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
        n_interests=N_INTERESTS,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    tag = "MetadataComiRecSA" if not use_label_aware else "MetadataComiRecSA+LabelAware+DivReg"

    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        total_rec = 0.0
        total_div = 0.0

        for batch in dl:
            for kk in batch:
                batch[kk] = batch[kk].to(DEVICE)

            loss, aux = label_aware_comirec_loss(
                model=model,
                batch=batch,
                label_tau=COMIREC_LABEL_TAU,
                diversity_reg_weight=COMIREC_DIVERSITY_REG,
                use_label_aware=use_label_aware,
            )

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += float(loss.item())
            total_rec += aux["rec_loss"]
            total_div += aux["div_loss"]

        r, n, m, cov = evaluate_multi_interest_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(
            f"[{tag}][seed={seed}] epoch {ep+1}/{EPOCHS}: "
            f"loss={total/max(1, len(dl)):.4f} "
            f"rec={total_rec/max(1, len(dl)):.4f} "
            f"div={total_div/max(1, len(dl)):.4f} | "
            f"valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}"
        )

        if stopper.step(n, model):
            print(f"[{tag}][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_multi_interest_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        "best_valid_NDCG@20": float(stopper.best if stopper.best is not None else n),
        **analyze_interest_usage(model, test_cases),
    }

    subset_rows = build_test_subset_rows_for_multi_interest_model(
        model_name=tag,
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows

## 13.1 Checkpointing and IRRT model-loading helpers

This section defines:
- checkpoint naming and saving
- model reconstruction from model names
- IRRT evaluation over saved checkpoints

The IRRT experiments reuse the exact trained model checkpoints from the main benchmark runs.

In [33]:
def slugify_model_name(name: str) -> str:
    return (
        name.replace("+", "_plus_")
        .replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
    )


def build_model_from_name(model_name: str, config: Dict[str, object]) -> nn.Module:
    if model_name == "GRU4Rec":
        return GRU4Rec(
            n_items=n_items,
            d_model=int(config["D_MODEL"]),
            hidden_size=int(config["GRU_HIDDEN"]),
            n_layers=1,
            dropout=float(config["DROPOUT"]),
        )

    if model_name in {"MetadataSASRec", "MetadataSASRec+CL4SRec", "MetadataSASRec+SimDCL"}:
        return MetadataSASRec(
            n_items=n_items,
            n_artists=N_ARTISTS,
            n_albums=N_ALBUMS,
            n_durations=N_DURATIONS,
            n_track_title_tokens=N_TRACK_TITLE_TOKENS,
            n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
            max_len=int(config["MAX_SEQ_LEN"]),
            d_model=int(config["D_MODEL"]),
            n_heads=int(config["N_HEADS"]),
            n_layers=int(config["N_LAYERS"]),
            dropout=float(config["DROPOUT"]),
        )

    if model_name == "FMLP-Rec":
        return FMLPRec(
            n_items=n_items,
            max_len=int(config["MAX_SEQ_LEN"]),
            d_model=int(config["D_MODEL"]),
            n_layers=int(config["FMLP_N_LAYERS"]),
            dropout=float(config["FMLP_DROPOUT"]),
        )

    if model_name in {"MetadataComiRecSA", "MetadataComiRecSA+LabelAware+DivReg"}:
        return MetadataComiRecSA(
            n_items=n_items,
            n_artists=N_ARTISTS,
            n_albums=N_ALBUMS,
            n_durations=N_DURATIONS,
            n_track_title_tokens=N_TRACK_TITLE_TOKENS,
            n_playlist_title_tokens=N_PLAYLIST_TITLE_TOKENS,
            max_len=int(config["MAX_SEQ_LEN"]),
            d_model=int(config["D_MODEL"]),
            n_heads=int(config["N_HEADS"]),
            n_layers=int(config["N_LAYERS"]),
            dropout=float(config["DROPOUT"]),
            n_interests=int(config["N_INTERESTS"]),
        )

    raise ValueError(f"Unsupported model_name for checkpoint loading: {model_name}")


def save_model_checkpoint(model: nn.Module, model_name: str, seed, checkpoint_path: Path) -> None:
    payload = {
        "model_name": model_name,
        "seed": seed,
        "config": dict(EXPERIMENT_CONFIG),
        "state_dict": model.state_dict(),
    }
    torch.save(payload, checkpoint_path)


def load_model_checkpoint(model_name: str, checkpoint_path: str) -> nn.Module:
    payload = torch.load(checkpoint_path, map_location=DEVICE)
    config = payload.get("config", dict(EXPERIMENT_CONFIG))
    state_dict = payload["state_dict"] if "state_dict" in payload else payload

    model = build_model_from_name(model_name, config).to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def run_irrt_for_checkpoint_row(
    row: pd.Series,
    eval_cases: EvalCases,
    dense_candidates: Dict[int, List[int]],
    sparse_candidates: Dict[int, List[int]],
    split_name: str,
) -> Tuple[List[Dict[str, object]], List[Dict[str, object]]]:
    model_name = row["model"]
    seed = row["seed"]
    checkpoint_path = row.get("checkpoint_path", "")

    if not checkpoint_path:
        return [], []

    model = load_model_checkpoint(model_name, checkpoint_path)

    hybrid_candidates = build_hybrid_candidate_dict(
        dense_dict=dense_candidates,
        sparse_dict=sparse_candidates,
        dense_weight=IRRT_HYBRID_DENSE_WEIGHT,
        sparse_weight=IRRT_HYBRID_SPARSE_WEIGHT,
        top_m=max(max(IRRT_RETRIEVAL_CUTOFFS), IRRT_RERANK_CANDIDATE_SIZE),
    )

    retrieval_rows = []
    for pipeline_name, cand_dict in [
        (EXACT_DENSE_PIPELINE_NAME, dense_candidates),
        (HYBRID_PIPELINE_NAME, hybrid_candidates),
    ]:
        rec_row = {
            "split": split_name,
            "model": model_name,
            "seed": seed,
            "pipeline": pipeline_name,
        }
        for m in IRRT_RETRIEVAL_CUTOFFS:
            rec_row[f"CandidateRecall@{m}"] = candidate_recall_at_m(cand_dict, eval_cases, m)
        retrieval_rows.append(rec_row)

    rerank_rows = []
    for pipeline_name, cand_dict in [
        ("SparseLexical->Rerank", sparse_candidates),
        (f"{HYBRID_PIPELINE_NAME}->Rerank", hybrid_candidates),
    ]:
        shortlist_dict = {u: cand_dict.get(u, [])[:IRRT_RERANK_CANDIDATE_SIZE] for u in eval_cases.users}

        reranked_dict = rerank_candidate_dict_with_model(
            model=model,
            eval_cases=eval_cases,
            candidate_dict=shortlist_dict,
            k=IRRT_RERANK_TOPK,
            eval_batch_size=EVAL_BATCH_SIZE,
        )

        metrics = compute_metrics_from_ranked_dict(
            ranked_dict=reranked_dict,
            eval_cases=eval_cases,
            n_items=n_items,
            k=IRRT_RERANK_TOPK,
        )

        rerank_rows.append({
            "split": split_name,
            "model": model_name,
            "seed": seed,
            "pipeline": pipeline_name,
            "candidate_size": IRRT_RERANK_CANDIDATE_SIZE,
            **metrics,
        })

    return retrieval_rows, rerank_rows

## 14. Main recommendation benchmark run

This section runs the main recommendation benchmark across:
- classic baselines
- metadata-aware sequential models
- SSL-enhanced metadata-aware models
- an efficient sequential alternative
- metadata-aware multi-interest models

For every neural run, the notebook also saves the trained checkpoint.
These checkpoints are later reused in the IRRT experiments.

In [ ]:
results = []
subset_results = []

top_valid_r, top_valid_n, top_valid_m, top_valid_cov = evaluate_toppop_full_ranking(
    valid_cases,
    valid_seqs,
    pop,
    n_items,
    TOPK,
)

experiment_plan = [
    ("TopPop", "fixed", lambda: (
        None,
        {
            "Recall@20": top_r,
            "NDCG@20": top_n,
            "MRR@20": top_m,
            "Coverage@20": top_cov,
            "best_valid_NDCG@20": top_valid_n,
        },
        build_test_subset_rows_for_toppop(
            model_name="TopPop",
            seed="fixed",
            eval_cases=test_cases,
            eval_seqs=test_seqs,
            pop_counter=pop,
            n_items=n_items,
            subset_masks=TEST_SUBSET_MASKS,
            k=TOPK,
        ),
    ))
]

for seed in SEEDS:
    experiment_plan.append(
        ("GRU4Rec", seed, lambda seed=seed: train_gru4rec(train_seqs, valid_cases, test_cases, seed))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("MetadataSASRec", seed, lambda seed=seed: train_metadata_sasrec(
            train_seqs, valid_cases, test_cases, seed, ssl_mode=None
        ))
    )

if DO_CL4SREC_SSL:
    for seed in SEEDS:
        experiment_plan.append(
            ("MetadataSASRec+CL4SRec", seed, lambda seed=seed: train_metadata_sasrec(
                train_seqs, valid_cases, test_cases, seed, ssl_mode="cl4srec"
            ))
        )

if DO_SIMDCL:
    for seed in SEEDS:
        experiment_plan.append(
            ("MetadataSASRec+SimDCL", seed, lambda seed=seed: train_metadata_sasrec(
                train_seqs, valid_cases, test_cases, seed, ssl_mode="simdcl"
            ))
        )

for seed in SEEDS:
    experiment_plan.append(
        ("FMLP-Rec", seed, lambda seed=seed: train_fmlprec(train_seqs, valid_cases, test_cases, seed))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("MetadataComiRecSA", seed, lambda seed=seed: train_comirec_sa_style(
            train_seqs, valid_cases, test_cases, seed, use_label_aware=False
        ))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("MetadataComiRecSA+LabelAware+DivReg", seed, lambda seed=seed: train_comirec_sa_style(
            train_seqs, valid_cases, test_cases, seed, use_label_aware=True
        ))
    )

print(f"Total runs: {len(experiment_plan)}")

for run_idx, (model_name, seed, run_fn) in enumerate(tqdm(experiment_plan, desc="Experiments"), start=1):
    print("=" * 80)
    print(f"START [{run_idx}/{len(experiment_plan)}] model={model_name}, seed={seed}")

    t0 = perf_counter()
    model_obj, metrics, subset_rows = run_fn()
    elapsed = perf_counter() - t0

    checkpoint_path = ""
    if model_obj is not None:
        ckpt_name = f"{slugify_model_name(model_name)}__seed_{seed}.pt"
        ckpt_path_obj = MODEL_DIR / ckpt_name
        save_model_checkpoint(model_obj, model_name=model_name, seed=seed, checkpoint_path=ckpt_path_obj)
        checkpoint_path = str(ckpt_path_obj)

    row = {
        "model": model_name,
        "seed": seed,
        "runtime_sec": round(elapsed, 2),
        "checkpoint_path": checkpoint_path,
        **metrics,
    }
    results.append(row)
    subset_results.extend(subset_rows)

    results_df = pd.DataFrame(results)
    subset_results_df = pd.DataFrame(subset_results)

    results_df.to_csv(PROJECT_DIR / "results_full_ranking_checkpoint.csv", index=False)
    subset_results_df.to_csv(PROJECT_DIR / "results_subset_checkpoint.csv", index=False)

    print(f"DONE  [{run_idx}/{len(experiment_plan)}] model={model_name}, seed={seed}, time={elapsed:.2f}s")

    clear_output(wait=True)
    print(f"Completed runs: {run_idx}/{len(experiment_plan)}")
    display(results_df.sort_values(["model", "seed"], kind="stable").reset_index(drop=True))

results_df = pd.DataFrame(results).sort_values(["model", "seed"], kind="stable").reset_index(drop=True)
subset_results_df = pd.DataFrame(subset_results).sort_values(["model", "subset", "seed"], kind="stable").reset_index(drop=True)

display(results_df)
display(subset_results_df.head(30))

Completed runs: 1/22


,model,seed,runtime_sec,checkpoint_path,Recall@20,NDCG@20,MRR@20,Coverage@20,best_valid_NDCG@20
0,TopPop,fixed,5.48,,0.018469,0.00754,0.004519,0.006,0.007133


START [2/22] model=GRU4Rec, seed=42


## 14.1 IRRT evaluation over saved checkpoints

This section runs the IRRT experiments on the saved neural checkpoints.

For every trained checkpoint, the notebook evaluates:
- sparse lexical retrieval
- dense retrieval from learned representations
- hybrid sparse+dense retrieval with reciprocal-rank fusion
- reranking of sparse, dense, and hybrid shortlists

The evaluation is computed on both validation and test splits, and then aggregated across seeds.
This avoids test-time model selection leakage.

In [ ]:
irrt_retrieval_rows = []
irrt_rerank_rows = []

neural_results_df = results_df[results_df["checkpoint_path"].fillna("") != ""].copy().reset_index(drop=True)

top_m_max = max(max(IRRT_RETRIEVAL_CUTOFFS), IRRT_RERANK_CANDIDATE_SIZE)

print("Precomputing model-independent sparse retrieval...")
valid_sparse_candidates = build_sparse_candidate_dict(
    eval_cases=valid_cases,
    eval_seqs=valid_seqs,
    top_m=top_m_max,
    title_weight=IRRT_TITLE_QUERY_WEIGHT,
    history_weight=IRRT_HISTORY_QUERY_WEIGHT,
    history_last_n=IRRT_HISTORY_TEXT_LAST_N,
)
test_sparse_candidates = build_sparse_candidate_dict(
    eval_cases=test_cases,
    eval_seqs=test_seqs,
    top_m=top_m_max,
    title_weight=IRRT_TITLE_QUERY_WEIGHT,
    history_weight=IRRT_HISTORY_QUERY_WEIGHT,
    history_last_n=IRRT_HISTORY_TEXT_LAST_N,
)

sparse_retrieval_rows = []
for split_name, eval_cases_obj, sparse_dict in [
    ("valid", valid_cases, valid_sparse_candidates),
    ("test", test_cases, test_sparse_candidates),
]:
    row = {
        "split": split_name,
        "pipeline": "SparseLexical",
    }
    for m in IRRT_RETRIEVAL_CUTOFFS:
        row[f"CandidateRecall@{m}"] = candidate_recall_at_m(sparse_dict, eval_cases_obj, m)
    sparse_retrieval_rows.append(row)

sparse_retrieval_df = pd.DataFrame(sparse_retrieval_rows)
display(sparse_retrieval_df)

for _, row in tqdm(neural_results_df.iterrows(), total=len(neural_results_df), desc="IRRT over checkpoints"):
    model = load_model_checkpoint(row["model"], row["checkpoint_path"])

    valid_dense_candidates = build_dense_candidate_dict(
        model=model,
        eval_cases=valid_cases,
        eval_seqs=valid_seqs,
        top_m=top_m_max,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )
    test_dense_candidates = build_dense_candidate_dict(
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        top_m=top_m_max,
        eval_batch_size=EVAL_BATCH_SIZE,
        item_chunk_size=ITEM_CHUNK_SIZE,
    )

    valid_ret_rows, valid_rerank_rows = run_irrt_for_checkpoint_row(
        row=row,
        eval_cases=valid_cases,
        dense_candidates=valid_dense_candidates,
        sparse_candidates=valid_sparse_candidates,
        split_name="valid",
    )
    test_ret_rows, test_rerank_rows = run_irrt_for_checkpoint_row(
        row=row,
        eval_cases=test_cases,
        dense_candidates=test_dense_candidates,
        sparse_candidates=test_sparse_candidates,
        split_name="test",
    )

    irrt_retrieval_rows.extend(valid_ret_rows)
    irrt_retrieval_rows.extend(test_ret_rows)

    irrt_rerank_rows.extend(valid_rerank_rows)
    irrt_rerank_rows.extend(test_rerank_rows)

irrt_retrieval_df = pd.DataFrame(irrt_retrieval_rows).sort_values(
    ["split", "pipeline", "model", "seed"], kind="stable"
).reset_index(drop=True)

irrt_rerank_df = pd.DataFrame(irrt_rerank_rows).sort_values(
    ["split", "pipeline", "model", "seed"], kind="stable"
).reset_index(drop=True)

sparse_retrieval_df.to_csv(PROJECT_DIR / "irrt_sparse_retrieval_results.csv", index=False)
irrt_retrieval_df.to_csv(PROJECT_DIR / "irrt_retrieval_results.csv", index=False)
irrt_rerank_df.to_csv(PROJECT_DIR / "irrt_rerank_results.csv", index=False)

display(sparse_retrieval_df)
display(irrt_retrieval_df.head(30))
display(irrt_rerank_df.head(30))

Precomputing model-independent sparse retrieval...


,split,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,valid,SparseLexical,0.548387,0.568548,0.568548
1,test,SparseLexical,0.584677,0.592742,0.592742


IRRT over checkpoints:   0%|          | 0/14 [00:00<?, ?it/s]

,split,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,valid,SparseLexical,0.548387,0.568548,0.568548
1,test,SparseLexical,0.584677,0.592742,0.592742


,split,model,seed,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,test,FMLP-Rec,42,ExactDenseTopM,0.774194,0.963710,0.963710
1,test,FMLP-Rec,43,ExactDenseTopM,0.766129,0.963710,0.963710
2,test,GRU4Rec,42,ExactDenseTopM,0.850806,0.963710,0.963710
3,test,GRU4Rec,43,ExactDenseTopM,0.846774,0.963710,0.963710
4,test,MetadataComiRecSA,42,ExactDenseTopM,0.834677,0.963710,0.963710
5,test,MetadataComiRecSA,43,ExactDenseTopM,0.798387,0.963710,0.963710
6,test,MetadataComiRecSA+LabelAware+DivReg,42,ExactDenseTopM,0.842742,0.963710,0.963710
7,test,MetadataComiRecSA+LabelAware+DivReg,43,ExactDenseTopM,0.806452,0.963710,0.963710
8,test,MetadataSASRec,42,ExactDenseTopM,0.826613,0.963710,0.963710
9,test,MetadataSASRec,43,ExactDenseTopM,0.806452,0.963710,0.963710


,split,model,seed,pipeline,candidate_size,Recall@20,NDCG@20,MRR@20,Coverage@20
0,test,FMLP-Rec,42,HybridRRF->Rerank,200,0.419355,0.172483,0.104488,0.97
1,test,FMLP-Rec,43,HybridRRF->Rerank,200,0.443548,0.178979,0.107207,0.96
2,test,GRU4Rec,42,HybridRRF->Rerank,200,0.584677,0.259808,0.168884,0.98
3,test,GRU4Rec,43,HybridRRF->Rerank,200,0.560484,0.237578,0.148003,0.97
4,test,MetadataComiRecSA,42,HybridRRF->Rerank,200,0.516129,0.233873,0.154918,0.99
5,test,MetadataComiRecSA,43,HybridRRF->Rerank,200,0.540323,0.231723,0.146803,1.00
6,test,MetadataComiRecSA+LabelAware+DivReg,42,HybridRRF->Rerank,200,0.524194,0.236758,0.156691,0.99
7,test,MetadataComiRecSA+LabelAware+DivReg,43,HybridRRF->Rerank,200,0.540323,0.238895,0.155194,0.99
8,test,MetadataSASRec,42,HybridRRF->Rerank,200,0.495968,0.213188,0.134525,0.98
9,test,MetadataSASRec,43,HybridRRF->Rerank,200,0.500000,0.205789,0.126479,0.95


## 15. Aggregate recommendation and IRRT results across seeds

After all runs finish, I summarize:
- full-ranking recommendation metrics
- subset metrics
- IRRT retrieval candidate recall
- IRRT reranked shortlist quality
- interest-usage statistics

All neural results are aggregated across seeds with mean values and confidence intervals.

In [ ]:
def summarize_metric(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    n = len(x)
    mean = x.mean()
    std = x.std(ddof=1) if n > 1 else 0.0
    ci95 = 1.96 * std / math.sqrt(n) if n > 1 else 0.0
    return pd.Series({"mean": mean, "std": std, "ci95": ci95, "n_seeds": n})


main_metric_cols = ["Recall@20", "NDCG@20", "MRR@20", "Coverage@20"]
interest_metric_cols = [
    "interest_usage_entropy",
    "interest_usage_max_share",
    "interest_target_score_entropy_mean",
]
summary_rows = []

for model_name, g in results_df.groupby("model", sort=False):
    row = {"model": model_name}

    if model_name == "TopPop":
        for col in main_metric_cols:
            row[col] = f"{float(g[col].iloc[0]):.4f}"
        row["best_valid_NDCG@20"] = f"{float(g['best_valid_NDCG@20'].iloc[0]):.4f}"
        row["n_seeds"] = 1
    else:
        for col in main_metric_cols:
            s = summarize_metric(g[col])
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"
        
        for col in interest_metric_cols:
            if col in g.columns:
                vals = g[col].dropna()
                if len(vals) == 0:
                    continue
                if len(vals) == 1:
                    row[col] = f"{float(vals.iloc[0]):.4f}"
                else:
                    s = summarize_metric(vals)
                    row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

        s_valid = summarize_metric(g["best_valid_NDCG@20"])
        row["best_valid_NDCG@20"] = f"{s_valid['mean']:.4f} ± {s_valid['ci95']:.4f}"
        row["n_seeds"] = int(s_valid["n_seeds"])

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,model,Recall@20,NDCG@20,MRR@20,Coverage@20,best_valid_NDCG@20,n_seeds,interest_usage_entropy,interest_usage_max_share,interest_target_score_entropy_mean
0,FMLP-Rec,0.4315 ± 0.0237,0.1757 ± 0.0064,0.1058 ± 0.0027,0.9650 ± 0.0098,0.2109 ± 0.0095,2,NaN,NaN,NaN
1,GRU4Rec,0.5726 ± 0.0237,0.2487 ± 0.0218,0.1584 ± 0.0205,0.9750 ± 0.0098,0.2667 ± 0.0236,2,NaN,NaN,NaN
2,MetadataComiRecSA,0.5282 ± 0.0237,0.2328 ± 0.0021,0.1509 ± 0.0080,0.9950 ± 0.0098,0.2258 ± 0.0058,2,0.9851 ± 0.1864,0.6492 ± 0.0948,1.3810 ± 0.0092
3,MetadataComiRecSA+LabelAware+DivReg,0.5323 ± 0.0158,0.2378 ± 0.0021,0.1559 ± 0.0015,0.9900 ± 0.0000,0.2258 ± 0.0017,2,0.8975 ± 0.0977,0.6552 ± 0.0356,1.3749 ± 0.0199
4,MetadataSASRec,0.4980 ± 0.0040,0.2095 ± 0.0073,0.1305 ± 0.0079,0.9650 ± 0.0294,0.2310 ± 0.0099,2,NaN,NaN,NaN
5,MetadataSASRec+CL4SRec,0.5222 ± 0.0593,0.2225 ± 0.0191,0.1399 ± 0.0074,0.9550 ± 0.0098,0.2477 ± 0.0108,2,NaN,NaN,NaN
6,MetadataSASRec+SimDCL,0.4960 ± 0.0395,0.2066 ± 0.0151,0.1280 ± 0.0086,0.9700 ± 0.0000,0.2242 ± 0.0023,2,NaN,NaN,NaN
7,TopPop,0.3508,0.1485,0.0925,0.5600,0.1499,1,NaN,NaN,NaN


In [ ]:
subset_summary_rows = []

for (model_name, subset_name), g in subset_results_df.groupby(["model", "subset"], sort=False):
    row = {
        "model": model_name,
        "subset": subset_name,
        "n_cases": int(g["n_cases"].iloc[0]),
    }

    for col in [f"Recall@{TOPK}", f"NDCG@{TOPK}", f"MRR@{TOPK}", f"Coverage@{TOPK}"]:
        vals = g[col].astype(float)
        if len(vals) == 1:
            row[col] = f"{float(vals.iloc[0]):.4f}"
        else:
            s = summarize_metric(vals)
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

    subset_summary_rows.append(row)

subset_summary_df = pd.DataFrame(subset_summary_rows)
display(subset_summary_df.sort_values(["subset", "model"]).reset_index(drop=True))

,model,subset,n_cases,Recall@20,NDCG@20,MRR@20,Coverage@20
0,FMLP-Rec,all,248,0.4315 ± 0.0237,0.1757 ± 0.0064,0.1058 ± 0.0027,0.9650 ± 0.0098
1,GRU4Rec,all,248,0.5726 ± 0.0237,0.2487 ± 0.0218,0.1584 ± 0.0205,0.9750 ± 0.0098
2,MetadataComiRecSA,all,248,0.5282 ± 0.0237,0.2328 ± 0.0021,0.1509 ± 0.0080,0.9950 ± 0.0098
3,MetadataComiRecSA+LabelAware+DivReg,all,248,0.5323 ± 0.0158,0.2378 ± 0.0021,0.1559 ± 0.0015,0.9900 ± 0.0000
4,MetadataSASRec,all,248,0.4980 ± 0.0040,0.2095 ± 0.0073,0.1305 ± 0.0079,0.9650 ± 0.0294
...,...,...,...,...,...,...,...
107,MetadataComiRecSA+LabelAware+DivReg,target_unseen,239,0.5523 ± 0.0164,0.2468 ± 0.0022,0.1618 ± 0.0015,0.9900 ± 0.0000
108,MetadataSASRec,target_unseen,239,0.5167 ± 0.0041,0.2174 ± 0.0075,0.1354 ± 0.0082,0.9650 ± 0.0294
109,MetadataSASRec+CL4SRec,target_unseen,239,0.5418 ± 0.0615,0.2309 ± 0.0199,0.1452 ± 0.0077,0.9550 ± 0.0098
110,MetadataSASRec+SimDCL,target_unseen,239,0.5146 ± 0.0410,0.2144 ± 0.0157,0.1328 ± 0.0089,0.9700 ± 0.0000


In [ ]:
subset_recall_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"Recall@{TOPK}",
    aggfunc="mean",
)

subset_ndcg_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"NDCG@{TOPK}",
    aggfunc="mean",
)

subset_mrr_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"MRR@{TOPK}",
    aggfunc="mean",
)

display(subset_recall_pivot)
display(subset_ndcg_pivot)
display(subset_mrr_pivot)

model,FMLP-Rec,GRU4Rec,MetadataComiRecSA,MetadataComiRecSA+LabelAware+DivReg,MetadataSASRec,MetadataSASRec+CL4SRec,MetadataSASRec+SimDCL,TopPop
subset,,,,,,,,
all,0.431452,0.572581,0.528226,0.532258,0.497984,0.522177,0.495968,0.350806
all_nonrepeat,0.447699,0.594142,0.548117,0.552301,0.516736,0.541841,0.514644,0.364017
div_heterogeneous,0.424051,0.512658,0.506329,0.512658,0.493671,0.531646,0.531646,0.303797
div_homogeneous,0.405063,0.658228,0.544304,0.556962,0.468354,0.493671,0.462025,0.329114
div_medium,0.512346,0.611111,0.592593,0.586420,0.586420,0.598765,0.549383,0.456790
hard_continuation,0.447699,0.594142,0.548117,0.552301,0.516736,0.541841,0.514644,0.364017
len_long,0.476744,0.598837,0.587209,0.575581,0.540698,0.604651,0.546512,0.383721
len_medium,0.381818,0.600000,0.527273,0.536364,0.500000,0.536364,0.490909,0.363636
len_short,0.459184,0.586735,0.525510,0.540816,0.505102,0.489796,0.500000,0.346939


model,FMLP-Rec,GRU4Rec,MetadataComiRecSA,MetadataComiRecSA+LabelAware+DivReg,MetadataSASRec,MetadataSASRec+CL4SRec,MetadataSASRec+SimDCL,TopPop
subset,,,,,,,,
all,0.175731,0.248693,0.232798,0.237827,0.209488,0.222549,0.206616,0.148521
all_nonrepeat,0.182348,0.258058,0.241565,0.246782,0.217377,0.230929,0.214396,0.154114
div_heterogeneous,0.147560,0.222604,0.215954,0.224628,0.196772,0.204592,0.198722,0.119634
div_homogeneous,0.178075,0.261564,0.246742,0.251638,0.206217,0.224137,0.216240,0.152947
div_medium,0.220446,0.289216,0.261493,0.263654,0.248357,0.263240,0.227886,0.188880
hard_continuation,0.182348,0.258058,0.241565,0.246782,0.217377,0.230929,0.214396,0.154114
len_long,0.198996,0.268159,0.251468,0.255969,0.228698,0.252352,0.232159,0.162166
len_medium,0.146449,0.244104,0.252819,0.248029,0.187782,0.229223,0.183416,0.161537
len_short,0.187887,0.257025,0.226557,0.238021,0.224051,0.213087,0.216195,0.142881


model,FMLP-Rec,GRU4Rec,MetadataComiRecSA,MetadataComiRecSA+LabelAware+DivReg,MetadataSASRec,MetadataSASRec+CL4SRec,MetadataSASRec+SimDCL,TopPop
subset,,,,,,,,
all,0.105848,0.158443,0.150861,0.155943,0.130502,0.139935,0.128028,0.092537
all_nonrepeat,0.109834,0.164410,0.156541,0.161815,0.135416,0.145205,0.132850,0.096021
div_heterogeneous,0.074518,0.141888,0.136695,0.145688,0.116224,0.115278,0.110317,0.070281
div_homogeneous,0.114227,0.151874,0.163037,0.166445,0.133184,0.149273,0.147530,0.103661
div_medium,0.139992,0.198602,0.169563,0.173028,0.156312,0.170425,0.140508,0.113675
hard_continuation,0.109834,0.164410,0.156541,0.161815,0.135416,0.145205,0.132850,0.096021
len_long,0.122056,0.174491,0.159391,0.166899,0.143399,0.154562,0.146759,0.099242
len_medium,0.083283,0.144035,0.177432,0.168155,0.102814,0.145654,0.102136,0.106450
len_short,0.114009,0.166998,0.142316,0.153795,0.146708,0.136742,0.137881,0.087342


## 15.1 Aggregate IRRT retrieval results

This section summarizes retrieval candidate recall for:
- sparse lexical retrieval
- dense retrieval
- hybrid sparse+dense retrieval
across validation and test splits.

In [ ]:
print("Model-independent sparse retrieval summary:")
display(sparse_retrieval_df)

irrt_retrieval_summary_rows = []

for (split_name, model_name, pipeline_name), g in irrt_retrieval_df.groupby(["split", "model", "pipeline"], sort=False):
    row = {
        "split": split_name,
        "model": model_name,
        "pipeline": pipeline_name,
    }

    for m in IRRT_RETRIEVAL_CUTOFFS:
        col = f"CandidateRecall@{m}"
        vals = g[col].astype(float)
        if len(vals) == 1:
            row[col] = f"{float(vals.iloc[0]):.4f}"
        else:
            s = summarize_metric(vals)
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

    irrt_retrieval_summary_rows.append(row)

irrt_retrieval_summary_df = pd.DataFrame(irrt_retrieval_summary_rows).sort_values(
    ["split", "pipeline", "model"], kind="stable"
).reset_index(drop=True)

display(irrt_retrieval_summary_df)

Model-independent sparse retrieval summary:


,split,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,valid,SparseLexical,0.548387,0.568548,0.568548
1,test,SparseLexical,0.584677,0.592742,0.592742


,split,model,pipeline,CandidateRecall@50,CandidateRecall@100,CandidateRecall@200
0,test,FMLP-Rec,ExactDenseTopM,0.7702 ± 0.0079,0.9637 ± 0.0000,0.9637 ± 0.0000
1,test,GRU4Rec,ExactDenseTopM,0.8488 ± 0.0040,0.9637 ± 0.0000,0.9637 ± 0.0000
2,test,MetadataComiRecSA,ExactDenseTopM,0.8165 ± 0.0356,0.9637 ± 0.0000,0.9637 ± 0.0000
3,test,MetadataComiRecSA+LabelAware+DivReg,ExactDenseTopM,0.8246 ± 0.0356,0.9637 ± 0.0000,0.9637 ± 0.0000
4,test,MetadataSASRec,ExactDenseTopM,0.8165 ± 0.0198,0.9637 ± 0.0000,0.9637 ± 0.0000
5,test,MetadataSASRec+CL4SRec,ExactDenseTopM,0.8468 ± 0.0474,0.9637 ± 0.0000,0.9637 ± 0.0000
6,test,MetadataSASRec+SimDCL,ExactDenseTopM,0.8024 ± 0.0158,0.9637 ± 0.0000,0.9637 ± 0.0000
7,test,FMLP-Rec,HybridRRF,0.7520 ± 0.0040,0.9637 ± 0.0000,0.9637 ± 0.0000
8,test,GRU4Rec,HybridRRF,0.8226 ± 0.0237,0.9637 ± 0.0000,0.9637 ± 0.0000
9,test,MetadataComiRecSA,HybridRRF,0.7843 ± 0.0040,0.9637 ± 0.0000,0.9637 ± 0.0000


## 15.2 Aggregate IRRT reranking results

This section summarizes final reranked shortlist quality for:
- sparse shortlist reranking
- dense shortlist reranking
- hybrid shortlist reranking
across validation and test splits.

In [ ]:
irrt_rerank_summary_rows = []

for (split_name, model_name, pipeline_name, candidate_size), g in irrt_rerank_df.groupby(
    ["split", "model", "pipeline", "candidate_size"], sort=False
):
    row = {
        "split": split_name,
        "model": model_name,
        "pipeline": pipeline_name,
        "candidate_size": int(candidate_size),
    }

    for col in [f"Recall@{IRRT_RERANK_TOPK}", f"NDCG@{IRRT_RERANK_TOPK}", f"MRR@{IRRT_RERANK_TOPK}", f"Coverage@{IRRT_RERANK_TOPK}"]:
        vals = g[col].astype(float)
        if len(vals) == 1:
            row[col] = f"{float(vals.iloc[0]):.4f}"
        else:
            s = summarize_metric(vals)
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

    irrt_rerank_summary_rows.append(row)

irrt_rerank_summary_df = pd.DataFrame(irrt_rerank_summary_rows).sort_values(
    ["split", "pipeline", "model"], kind="stable"
).reset_index(drop=True)

display(irrt_rerank_summary_df)

,split,model,pipeline,candidate_size,Recall@20,NDCG@20,MRR@20,Coverage@20
0,test,FMLP-Rec,HybridRRF->Rerank,200,0.4315 ± 0.0237,0.1757 ± 0.0064,0.1058 ± 0.0027,0.9650 ± 0.0098
1,test,GRU4Rec,HybridRRF->Rerank,200,0.5726 ± 0.0237,0.2487 ± 0.0218,0.1584 ± 0.0205,0.9750 ± 0.0098
2,test,MetadataComiRecSA,HybridRRF->Rerank,200,0.5282 ± 0.0237,0.2328 ± 0.0021,0.1509 ± 0.0080,0.9950 ± 0.0098
3,test,MetadataComiRecSA+LabelAware+DivReg,HybridRRF->Rerank,200,0.5323 ± 0.0158,0.2378 ± 0.0021,0.1559 ± 0.0015,0.9900 ± 0.0000
4,test,MetadataSASRec,HybridRRF->Rerank,200,0.4980 ± 0.0040,0.2095 ± 0.0073,0.1305 ± 0.0079,0.9650 ± 0.0294
5,test,MetadataSASRec+CL4SRec,HybridRRF->Rerank,200,0.5222 ± 0.0593,0.2225 ± 0.0191,0.1399 ± 0.0074,0.9550 ± 0.0098
6,test,MetadataSASRec+SimDCL,HybridRRF->Rerank,200,0.4960 ± 0.0395,0.2066 ± 0.0151,0.1280 ± 0.0086,0.9700 ± 0.0000
7,test,FMLP-Rec,SparseLexical->Rerank,200,0.3992 ± 0.0237,0.1602 ± 0.0145,0.0951 ± 0.0115,0.9350 ± 0.0098
8,test,GRU4Rec,SparseLexical->Rerank,200,0.4637 ± 0.0474,0.2109 ± 0.0064,0.1398 ± 0.0037,0.9450 ± 0.0098
9,test,MetadataComiRecSA,SparseLexical->Rerank,200,0.4435 ± 0.0079,0.1961 ± 0.0056,0.1273 ± 0.0066,0.9450 ± 0.0098
